In [1]:
# -*- coding: utf-8 -*-
"""
多模型投票搜索最优 ug/mL 阈值（扩展版，>=10个模型）

逻辑：
1) 固定 uM 阈值 = 50% 分位数（中位数）
2) 遍历所有 ug/mL 候选阈值
3) 对每个候选阈值，用多个模型在 ug/mL 子集上训练
4) 再到 uM 子集上验证，与 uM 参考标签比较
5) 每个模型独立给出阈值排名
6) 用“倒数排名投票（reciprocal-rank voting）”做综合投票
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

# 可选：XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 1) 路径与参数
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"

OUT_DIR = Path(r"C:\Users\86158\Desktop\threshold_search_multi_model")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHEET_NAME = 0
COL_IC50 = "IC50/μM"

# 固定 uM 阈值：50%分位数（中位数）
UM_REF_QUANTILE = 0.5

# ug/mL 每个候选阈值切分后，两类至少要有几个样本
MIN_CLASS_COUNT = 2

# 投票时是否只看每个模型前K名（None 表示看全部）
TOP_K_FOR_VOTE = None

# 输出文件
OUT_FULL_RESULTS = OUT_DIR / "all_models_all_thresholds.csv"
OUT_MODEL_BEST = OUT_DIR / "best_threshold_per_model.csv"
OUT_VOTE = OUT_DIR / "threshold_vote_summary.csv"
OUT_FEATURE_INFO = OUT_DIR / "feature_processing_info.csv"


# =========================
# 2) 解析 IC50
# =========================
def parse_ic50_cell(x):
    """
    解析 IC50：
    - 纯数字 -> 默认 uM
    - '数值 + uM/μM/µM' -> uM
    - '数值 + ug/mL/μg/mL/µg/mL' -> ug/mL
    - 其它 -> unparsed
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")

    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"

    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================
# 3) 特征处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    """
    分类列统一清洗：
    - 转字符串
    - 去首尾空格
    - 合并中间多余空格
    - 小写化
    - 空字符串 / nan / none 视作缺失
    """
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    })
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    # 数值列：强制转数值
    for c in numeric_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    # 分类列：统一清洗
    for c in categorical_cols:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess(numeric_cols, categorical_cols):
    # 用 dense 输出，兼容 GaussianNB / HistGB / GradientBoosting 等
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )
    return preprocess


# =========================
# 4) 模型定义
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ),

        "LinearSVC": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42
        ),

        "SVC_RBF": SVC(
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),

        "GaussianNB": GaussianNB(),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),

        "RF": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

        "HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )

    return models


def needs_scaling(model_name):
    """
    哪些模型更适合在 one-hot + numeric 拼接后再做标准化
    """
    return model_name in {
        "LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"
    }


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]

    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))

    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    """
    统一输出连续分数：
    - 若有 predict_proba，则取正类概率
    - 若只有 decision_function，则用 sigmoid 压到 (0,1)
    """
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]

    if hasattr(pipe, "decision_function"):
        z = pipe.decision_function(X)
        z = np.asarray(z, dtype=float)
        return 1.0 / (1.0 + np.exp(-z))

    pred = pipe.predict(X)
    return np.asarray(pred, dtype=float)


# =========================
# 5) 主程序
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    # ---- IC50 解析
    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    # ---- 明确特征列
    feature_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "shape", "size (nm)", "surface treatment", "dispersion medium",
        "pH", "temperature/oC", "Substrate "
    ]

    numeric_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]

    categorical_cols = [
        "shape", "surface treatment", "dispersion medium", "Substrate "
    ]

    # 防止列缺失
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"这些特征列不存在：{missing_cols}")

    # 保存列处理信息
    feature_info = pd.DataFrame({
        "feature": feature_cols,
        "type": ["numeric" if c in numeric_cols else "categorical" for c in feature_cols]
    })
    feature_info.to_csv(OUT_FEATURE_INFO, index=False, encoding="utf-8-sig")

    # ---- 特征矩阵
    X_all = prepare_features(df, feature_cols, numeric_cols, categorical_cols)
    preprocess = make_preprocess(numeric_cols, categorical_cols)

    # ---- 有效 IC50
    m_valid = (
        pd.to_numeric(df["IC50_value"], errors="coerce").notna() &
        (pd.to_numeric(df["IC50_value"], errors="coerce") > 0) &
        df["IC50_unit"].isin(["uM", "ug/mL"])
    )
    df_valid = df.loc[m_valid].copy()
    X_valid = X_all.loc[m_valid].copy()

    # ---- uM / ugmL 分组
    m_um = df_valid["IC50_unit"].eq("uM")
    m_ug = df_valid["IC50_unit"].eq("ug/mL")

    df_um = df_valid.loc[m_um].copy()
    df_ug = df_valid.loc[m_ug].copy()
    X_um = X_valid.loc[m_um].copy()
    X_ug = X_valid.loc[m_ug].copy()

    if len(df_um) == 0 or len(df_ug) == 0:
        raise RuntimeError("uM 或 ug/mL 有效样本为空，无法搜索阈值。")

    # ---- 固定 uM 阈值 = 50%分位数
    thr_um = float(np.quantile(df_um["IC50_value"].astype(float).values, UM_REF_QUANTILE))
    y_um = (df_um["IC50_value"].astype(float).values <= thr_um).astype(int)

    if (np.sum(y_um == 0) < MIN_CLASS_COUNT) or (np.sum(y_um == 1) < MIN_CLASS_COUNT):
        raise RuntimeError("uM 参考标签只有一类，无法评估 AUC/Kappa/MCC/ACC。")

    # ---- 遍历 ug/mL 候选阈值
    candidate_thresholds = np.sort(df_ug["IC50_value"].astype(float).unique())

    models = get_models()
    rows = []

    for model_name, model in models.items():
        print(f"\n[Running] {model_name} ...")

        for thr_ug in candidate_thresholds:
            y_ug = (df_ug["IC50_value"].astype(float).values <= thr_ug).astype(int)
            n0 = int(np.sum(y_ug == 0))
            n1 = int(np.sum(y_ug == 1))

            # 至少两类
            if n0 < MIN_CLASS_COUNT or n1 < MIN_CLASS_COUNT:
                continue

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(X_ug, y_ug)

                # 在 uM 上评估
                p_um = score_to_prob(pipe, X_um)
                pred_um = (p_um >= 0.5).astype(int)

                auc = roc_auc_score(y_um, p_um)
                kappa = cohen_kappa_score(y_um, pred_um)
                mcc = matthews_corrcoef(y_um, pred_um)
                acc = accuracy_score(y_um, pred_um)

                rows.append({
                    "model": model_name,
                    "thr_uM_fixed": float(thr_um),
                    "thr_ugmL_candidate": float(thr_ug),
                    "n_uM": int(len(df_um)),
                    "n_ugmL": int(len(df_ug)),
                    "uM_pos_rate": float(np.mean(y_um)),
                    "ugmL_pos_rate": float(np.mean(y_ug)),
                    "auc_uM": float(auc),
                    "kappa_uM": float(kappa),
                    "mcc_uM": float(mcc),
                    "acc_uM": float(acc),
                    "n_ugmL_class0": n0,
                    "n_ugmL_class1": n1,
                    "error": ""
                })

            except Exception as e:
                rows.append({
                    "model": model_name,
                    "thr_uM_fixed": float(thr_um),
                    "thr_ugmL_candidate": float(thr_ug),
                    "n_uM": int(len(df_um)),
                    "n_ugmL": int(len(df_ug)),
                    "uM_pos_rate": float(np.mean(y_um)),
                    "ugmL_pos_rate": float(np.mean(y_ug)),
                    "auc_uM": np.nan,
                    "kappa_uM": np.nan,
                    "mcc_uM": np.nan,
                    "acc_uM": np.nan,
                    "n_ugmL_class0": n0,
                    "n_ugmL_class1": n1,
                    "error": repr(e)
                })

    res = pd.DataFrame(rows)
    res.to_csv(OUT_FULL_RESULTS, index=False, encoding="utf-8-sig")

    res_ok = res.dropna(subset=["auc_uM", "kappa_uM", "mcc_uM", "acc_uM"]).copy()
    if res_ok.empty:
        print(res.to_string(index=False))
        raise RuntimeError("所有模型/阈值都失败了，请检查 error 列。")

    # =========================
    # 6) 每个模型独立排序并取最佳阈值
    # 排序规则：AUC > Kappa > MCC > ACC
    # =========================
    ranked_list = []
    best_list = []

    for model_name, sub in res_ok.groupby("model"):
        sub = sub.sort_values(
            by=["auc_uM", "kappa_uM", "mcc_uM", "acc_uM"],
            ascending=[False, False, False, False]
        ).reset_index(drop=True)

        sub["rank_in_model"] = np.arange(1, len(sub) + 1)
        ranked_list.append(sub)

        best_row = sub.iloc[0].copy()
        best_list.append(best_row)

    ranked_all = pd.concat(ranked_list, ignore_index=True)
    best_df = pd.DataFrame(best_list).sort_values(
        by=["auc_uM", "kappa_uM", "mcc_uM", "acc_uM"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    best_df.to_csv(OUT_MODEL_BEST, index=False, encoding="utf-8-sig")

    # =========================
    # 7) 多模型投票：倒数排名投票
    # =========================
    vote_rows = []

    for model_name, sub in ranked_all.groupby("model"):
        sub = sub.copy()

        if TOP_K_FOR_VOTE is not None:
            sub = sub[sub["rank_in_model"] <= TOP_K_FOR_VOTE].copy()

        sub["vote_score"] = 1.0 / sub["rank_in_model"].astype(float)
        sub["top1_flag"] = (sub["rank_in_model"] == 1).astype(int)
        sub["top3_flag"] = (sub["rank_in_model"] <= 3).astype(int)

        vote_rows.append(sub[[
            "model", "thr_ugmL_candidate", "rank_in_model",
            "vote_score", "top1_flag", "top3_flag",
            "auc_uM", "kappa_uM", "mcc_uM", "acc_uM"
        ]])

    vote_long = pd.concat(vote_rows, ignore_index=True)

    vote_summary = (
        vote_long.groupby("thr_ugmL_candidate", as_index=False)
        .agg(
            n_models_voted=("model", "nunique"),
            vote_score_sum=("vote_score", "sum"),
            top1_count=("top1_flag", "sum"),
            top3_count=("top3_flag", "sum"),
            mean_rank=("rank_in_model", "mean"),
            mean_auc=("auc_uM", "mean"),
            mean_kappa=("kappa_uM", "mean"),
            mean_mcc=("mcc_uM", "mean"),
            mean_acc=("acc_uM", "mean"),
        )
        .sort_values(
            by=["vote_score_sum", "top1_count", "top3_count", "mean_auc", "mean_kappa", "mean_mcc", "mean_acc"],
            ascending=[False, False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )

    vote_summary.to_csv(OUT_VOTE, index=False, encoding="utf-8-sig")

    consensus = vote_summary.iloc[0]
    best_single = best_df.iloc[0]

    # =========================
    # 8) 控制台输出
    # =========================
    print("\n" + "=" * 60)
    print("特征处理确认")
    print("=" * 60)
    print(f"总特征数: {len(feature_cols)}")
    print("\n数值特征：")
    for c in numeric_cols:
        print(" -", c)
    print("\n分类特征：")
    for c in categorical_cols:
        print(" -", c)

    print("\n" + "=" * 60)
    print("固定参考阈值")
    print("=" * 60)
    print(f"uM 固定阈值（50%分位数）: {thr_um:.6g}")
    print(f"uM 样本数: {len(df_um)}")
    print(f"ug/mL 样本数: {len(df_ug)}")
    print(f"模型数: {len(models)}")

    print("\n" + "=" * 60)
    print("每个模型的最佳 ug/mL 阈值")
    print("=" * 60)
    print(best_df[[
        "model", "thr_ugmL_candidate", "auc_uM", "kappa_uM", "mcc_uM", "acc_uM", "ugmL_pos_rate"
    ]].to_string(index=False))

    print("\n" + "=" * 60)
    print("多模型投票后的综合最优阈值")
    print("=" * 60)
    print(f"Consensus ug/mL threshold = {consensus['thr_ugmL_candidate']:.6g}")
    print(f"vote_score_sum = {consensus['vote_score_sum']:.4f}")
    print(f"top1_count     = {int(consensus['top1_count'])}")
    print(f"top3_count     = {int(consensus['top3_count'])}")
    print(f"mean_rank      = {consensus['mean_rank']:.4f}")
    print(f"mean_auc       = {consensus['mean_auc']:.4f}")
    print(f"mean_kappa     = {consensus['mean_kappa']:.4f}")
    print(f"mean_mcc       = {consensus['mean_mcc']:.4f}")
    print(f"mean_acc       = {consensus['mean_acc']:.4f}")

    print("\n" + "=" * 60)
    print("单模型里表现最好的结果（仅供参考）")
    print("=" * 60)
    print(f"Best single model = {best_single['model']}")
    print(f"Best single threshold = {best_single['thr_ugmL_candidate']:.6g}")
    print(f"AUC   = {best_single['auc_uM']:.4f}")
    print(f"Kappa = {best_single['kappa_uM']:.4f}")
    print(f"MCC   = {best_single['mcc_uM']:.4f}")
    print(f"ACC   = {best_single['acc_uM']:.4f}")

    print("\n" + "=" * 60)
    print("文件已保存")
    print("=" * 60)
    print(OUT_FEATURE_INFO)
    print(OUT_FULL_RESULTS)
    print(OUT_MODEL_BEST)
    print(OUT_VOTE)

    print("\nTop 10 投票结果：")
    print(vote_summary.head(10).to_string(index=False))
    print("\nTop 10 投票结果：")

if __name__ == "__main__":
    main()


[Running] LogReg ...

[Running] LinearSVC ...

[Running] SVC_RBF ...

[Running] KNN ...

[Running] GaussianNB ...

[Running] DecisionTree ...

[Running] RF ...

[Running] ExtraTrees ...

[Running] GradientBoosting ...

[Running] HistGB ...

[Running] AdaBoost ...

[Running] XGBoost ...

特征处理确认
总特征数: 27

数值特征：
 - contain O
 - contain N
 - contain P
 - contain S
 - contain Si
 - contain Se
 - contain B
 - contain F
 - contain Cl
 - contain Br
 - contain I
 - main  metal proportion
 - main metal number
 - main metal valence
 - minor metal percentage
 - minor metal number
 - minor metal valence
 - 3rd metal ratio
 - 3rd metal type
 - 3rd metal valence
 - size (nm)
 - pH
 - temperature/oC

分类特征：
 - shape
 - surface treatment
 - dispersion medium
 - Substrate 

固定参考阈值
uM 固定阈值（50%分位数）: 0.109
uM 样本数: 27
ug/mL 样本数: 23
模型数: 12

每个模型的最佳 ug/mL 阈值
           model  thr_ugmL_candidate   auc_uM  kappa_uM   mcc_uM   acc_uM  ugmL_pos_rate
         XGBoost              4.1000 0.818681  0.005666 0.01048

In [11]:
# -*- coding: utf-8 -*-
"""
SOD 粗筛分类模型训练（两个单位合并 + DOI分组切分 + 12模型比选）

核心规则：
1) 标签：
   - uM:    IC50 <= 0.109  -> 高活性(1)
   - ug/mL: IC50 <= 0.7    -> 高活性(1)
   - 备用ug/mL阈值仅记录，不参与本次训练：0.3, 0.384

2) 数据：
   - 合并 uM 与 ug/mL 两种单位
   - 只保留能解析且单位属于 {uM, ug/mL} 且 IC50 > 0 的样本

3) 分割：
   - 按 DOI 分组（同一个 DOI 只会出现在训练集或测试集）
   - 对缺失 DOI 的样本，给每行单独分配唯一伪 DOI，避免把无关样本绑成同一组

4) 模型：
   - LogReg
   - LinearSVC
   - SVC_RBF
   - KNN
   - GaussianNB
   - DecisionTree
   - RF
   - ExtraTrees
   - GradientBoosting
   - HistGB
   - AdaBoost
   - XGBoost（若本机装了 xgboost）

5) 模型选择：
   - 以分组重复切分后的 AUC_mean 为主
   - 再依次参考 AUC_median, MCC_mean, Kappa_mean, ACC_mean
   - 选最稳的最佳模型，而不是单次最高分模型

输出：
- sod_cls_grouped_labeled_data.csv
- sod_cls_grouped_feature_info.csv
- sod_cls_grouped_all_splits.csv
- sod_cls_grouped_summary.csv
- sod_cls_grouped_best_model_refit_full.joblib
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

# 可选：XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_cls_grouped_search")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IC50 = "IC50/μM"
COL_DOI = "data reference doi"
COL_NANOZYME = "nanozyme"

# 固定阈值
THR_IC50_UM = 0.109
THR_IC50_UGML = 0.7

# 备用阈值（只记录，不参与本次训练）
THR_IC50_UGML_BACKUPS = [0.3, 0.384]

# 分组切分设置
TEST_SIZE = 0.2
N_SPLITS = 50
SPLIT_SEED = 42
MIN_TEST_SAMPLES = 6
MIN_CLASS_COUNT = 2

# 输出文件
OUT_LABELED = OUT_DIR / "sod_cls_grouped_labeled_data.csv"
OUT_FEATURE_INFO = OUT_DIR / "sod_cls_grouped_feature_info.csv"
OUT_ALL_SPLITS = OUT_DIR / "sod_cls_grouped_all_splits.csv"
OUT_SUMMARY = OUT_DIR / "sod_cls_grouped_summary.csv"
OUT_BEST_MODEL = OUT_DIR / "sod_cls_grouped_best_model_refit_full.joblib"


# =========================
# 1) IC50 解析与标签
# =========================
def parse_ic50_cell(x):
    """
    解析 IC50：
    - 纯数字 -> 默认 uM
    - '数值 + uM/μM/µM' -> uM
    - '数值 + ug/mL/μg/mL/µg/mL' -> ug/mL
    - 其它 -> unparsed
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")

    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"

    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    """
    固定阈值标签：
    - uM:    <= 0.109 -> 1
    - ug/mL: <= 0.7   -> 1
    """
    if pd.isna(ic50_value):
        return np.nan

    try:
        v = float(ic50_value)
    except Exception:
        return np.nan

    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= THR_IC50_UM)
    elif ic50_unit == "ug/mL":
        return int(v <= THR_IC50_UGML)
    else:
        return np.nan


# =========================
# 2) 特征处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    """
    分类列清洗：
    - 转字符串
    - 去首尾空格
    - 合并中间多余空格
    - 小写化
    - 空字符串 / nan / none -> 缺失
    """
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    })
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    # 数值列强制转数值
    for c in numeric_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    # 分类列统一清洗
    for c in categorical_cols:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess(numeric_cols, categorical_cols):
    # 用 dense 输出，兼容 GaussianNB / HistGB / GradientBoosting 等
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )
    return preprocess


# =========================
# 3) 模型定义
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ),

        "LinearSVC": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42
        ),

        "SVC_RBF": SVC(
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),

        "GaussianNB": GaussianNB(),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),

        "RF": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

        "HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )

    return models


def needs_scaling(model_name):
    """
    更适合标准化的模型
    """
    return model_name in {
        "LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"
    }


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]

    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))

    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    """
    统一输出连续分数：
    - 若有 predict_proba，则取正类概率
    - 若只有 decision_function，则做 sigmoid 压缩到 (0,1)
    """
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]

    if hasattr(pipe, "decision_function"):
        z = pipe.decision_function(X)
        z = np.asarray(z, dtype=float)
        return 1.0 / (1.0 + np.exp(-z))

    pred = pipe.predict(X)
    return np.asarray(pred, dtype=float)


# =========================
# 4) 主流程
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    # ---- 解析 IC50
    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    # ---- 固定阈值打标签
    df["y_IC50Low"] = df.apply(
        lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]),
        axis=1
    )

    # ---- 明确特征列
    feature_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "shape", "size (nm)", "surface treatment", "dispersion medium",
        "pH", "temperature/oC", "Substrate "
    ]

    numeric_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]

    categorical_cols = [
        "shape", "surface treatment", "dispersion medium", "Substrate "
    ]

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"以下特征列在文件中不存在：{missing_cols}")

    # ---- 保存特征处理信息
    feature_info = pd.DataFrame({
        "feature": feature_cols,
        "type": ["numeric" if c in numeric_cols else "categorical" for c in feature_cols]
    })
    feature_info.to_csv(OUT_FEATURE_INFO, index=False, encoding="utf-8-sig")

    # ---- 仅保留有效标签样本
    m_valid = df["y_IC50Low"].notna()
    df_use = df.loc[m_valid].copy()

    if df_use.empty:
        raise RuntimeError("没有可用于 SOD 分类训练的有效样本。")

    # ---- 分组：同 DOI 只出现在 train/test 一侧
    # 缺失 DOI 的样本，给每条记录分配唯一伪 DOI，避免无关样本被绑在一起
    groups = df_use[COL_DOI].copy()
    groups = groups.fillna("").astype(str).str.strip()
    missing_mask = groups.eq("")
    groups.loc[missing_mask] = [
        f"__NO_DOI_ROW_{i}__" for i in df_use.index[missing_mask]
    ]
    groups = groups.values

    # ---- 特征矩阵与标签
    X = prepare_features(df_use, feature_cols, numeric_cols, categorical_cols)
    y = df_use["y_IC50Low"].astype(int).values
    preprocess = make_preprocess(numeric_cols, categorical_cols)

    # ---- 保存带标签数据，便于追溯
    df_use.to_csv(OUT_LABELED, index=False, encoding="utf-8-sig")

    # ---- 数据概况
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    print("=" * 60)
    print("SOD 粗筛分类训练数据概况")
    print("=" * 60)
    print(f"总样本数: {len(df_use)}")
    print(f"高活性(1): {n_pos}")
    print(f"低活性(0): {n_neg}")
    print(f"uM 阈值: {THR_IC50_UM}")
    print(f"ug/mL 阈值: {THR_IC50_UGML}")
    print(f"ug/mL 备用阈值记录: {THR_IC50_UGML_BACKUPS}")
    print(f"分组列: {COL_DOI}")
    print(f"唯一 group 数: {len(np.unique(groups))}")
    print(f"模型数: {len(get_models())}")

    # ---- 12模型 + 分组重复切分评估
    splitter = GroupShuffleSplit(
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        random_state=SPLIT_SEED
    )

    models = get_models()
    rows = []

    for model_name, model in models.items():
        print(f"\n[Running] {model_name} ...")

        for split_id, (tr, te) in enumerate(splitter.split(X, y, groups=groups), start=1):
            if len(te) < MIN_TEST_SAMPLES:
                continue

            ytr, yte = y[tr], y[te]

            # 训练集 / 测试集都要有两类
            if (np.sum(ytr == 0) < MIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_CLASS_COUNT):
                continue
            if (np.sum(yte == 0) < MIN_CLASS_COUNT) or (np.sum(yte == 1) < MIN_CLASS_COUNT):
                continue

            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(Xtr, ytr)

                pte = score_to_prob(pipe, Xte)
                pred_te = (pte >= 0.5).astype(int)

                auc = roc_auc_score(yte, pte)
                kappa = cohen_kappa_score(yte, pred_te)
                mcc = matthews_corrcoef(yte, pred_te)
                acc = accuracy_score(yte, pred_te)

                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "n_train": int(len(tr)),
                    "n_test": int(len(te)),
                    "pos_train": int(np.sum(ytr == 1)),
                    "neg_train": int(np.sum(ytr == 0)),
                    "pos_test": int(np.sum(yte == 1)),
                    "neg_test": int(np.sum(yte == 0)),
                    "auc": float(auc),
                    "kappa": float(kappa),
                    "mcc": float(mcc),
                    "acc": float(acc),
                    "error": ""
                })

            except Exception as e:
                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "n_train": int(len(tr)),
                    "n_test": int(len(te)),
                    "pos_train": int(np.sum(ytr == 1)),
                    "neg_train": int(np.sum(ytr == 0)),
                    "pos_test": int(np.sum(yte == 1)),
                    "neg_test": int(np.sum(yte == 0)),
                    "auc": np.nan,
                    "kappa": np.nan,
                    "mcc": np.nan,
                    "acc": np.nan,
                    "error": repr(e)
                })

    res = pd.DataFrame(rows)
    if res.empty:
        raise RuntimeError("没有得到任何有效分组评估结果。")

    res.to_csv(OUT_ALL_SPLITS, index=False, encoding="utf-8-sig")

    res_ok = res.dropna(subset=["auc", "kappa", "mcc", "acc"]).copy()
    if res_ok.empty:
        print(res.to_string(index=False))
        raise RuntimeError("所有模型评估都失败了，请检查 all_splits 文件中的 error 列。")

    # ---- 汇总：用稳定性优先，而不是 max 优先
    summary = (
        res_ok.groupby("model", as_index=False)
        .agg(
            auc_mean=("auc", "mean"),
            auc_median=("auc", "median"),
            auc_std=("auc", "std"),
            auc_max=("auc", "max"),
            kappa_mean=("kappa", "mean"),
            mcc_mean=("mcc", "mean"),
            acc_mean=("acc", "mean"),
            n_splits=("split", "count")
        )
        .sort_values(
            by=["auc_mean", "auc_median", "mcc_mean", "kappa_mean", "acc_mean", "auc_max"],
            ascending=[False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )

    summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    best_model_name = str(summary.iloc[0]["model"])
    print("\n" + "=" * 60)
    print("模型汇总（按稳定性排序）")
    print("=" * 60)
    print(summary.to_string(index=False))

    print("\n" + "=" * 60)
    print("最佳粗筛分类模型")
    print("=" * 60)
    print(f"Best model = {best_model_name}")
    print(f"AUC_mean   = {summary.iloc[0]['auc_mean']:.4f}")
    print(f"AUC_median = {summary.iloc[0]['auc_median']:.4f}")
    print(f"MCC_mean   = {summary.iloc[0]['mcc_mean']:.4f}")
    print(f"Kappa_mean = {summary.iloc[0]['kappa_mean']:.4f}")
    print(f"ACC_mean   = {summary.iloc[0]['acc_mean']:.4f}")
    print(f"n_splits   = {int(summary.iloc[0]['n_splits'])}")

    # ---- 用全部标注数据重训最佳模型
    best_model = get_models()[best_model_name]
    best_pipe = build_model_pipe(best_model_name, best_model, preprocess)
    best_pipe.fit(X, y)
    joblib.dump(best_pipe, OUT_BEST_MODEL)

    print("\n" + "=" * 60)
    print("文件已保存")
    print("=" * 60)
    print(OUT_LABELED)
    print(OUT_FEATURE_INFO)
    print(OUT_ALL_SPLITS)
    print(OUT_SUMMARY)
    print(OUT_BEST_MODEL)


if __name__ == "__main__":
    main()

SOD 粗筛分类训练数据概况
总样本数: 50
高活性(1): 24
低活性(0): 26
uM 阈值: 0.109
ug/mL 阈值: 0.7
ug/mL 备用阈值记录: [0.3, 0.384]
分组列: data reference doi
唯一 group 数: 24
模型数: 12

[Running] LogReg ...

[Running] LinearSVC ...

[Running] SVC_RBF ...

[Running] KNN ...

[Running] GaussianNB ...

[Running] DecisionTree ...

[Running] RF ...

[Running] ExtraTrees ...

[Running] GradientBoosting ...

[Running] HistGB ...

[Running] AdaBoost ...

[Running] XGBoost ...

模型汇总（按稳定性排序）
           model  auc_mean  auc_median  auc_std  auc_max  kappa_mean  mcc_mean  acc_mean  n_splits
         XGBoost  0.674571    0.666667 0.183574 1.000000    0.159702  0.179220  0.578355        47
              RF  0.659233    0.656250 0.195767 1.000000    0.093948  0.104562  0.557168        47
GradientBoosting  0.628855    0.642857 0.205969 1.000000    0.106607  0.120973  0.558052        47
      ExtraTrees  0.625908    0.628571 0.179650 1.000000    0.084181  0.091199  0.535857        47
          LogReg  0.623255    0.653061 0.190562 1.000000

In [12]:
# -*- coding: utf-8 -*-
"""
SOD 0.7 + XGBoost 分类模型的 SHAP 分析
输出：
1) SHAP beeswarm 图
2) SHAP 平均绝对值条形图
3) 聚合到原始特征层面的 SHAP 重要性图
4) XGBoost 内置 gain / split count 特征重要性图
5) 各类 CSV 表，便于后续构效关系分析
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, cohen_kappa_score
from xgboost import XGBClassifier


# =========================
# 0) 路径与参数
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_xgb_shap_0p7")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IC50 = "IC50/μM"

# 你当前确定的阈值
THR_IC50_UM = 0.109
THR_IC50_UGML = 0.7


# =========================
# 1) IC50 解析与标签
# =========================
def parse_ic50_cell(x):
    """
    解析 IC50：
    - 纯数字 -> 默认 uM
    - '数值 + uM/μM/µM' -> uM
    - '数值 + ug/mL/μg/mL/µg/mL' -> ug/mL
    - 其它 -> unparsed
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")

    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"

    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    """
    固定阈值标签：
    - uM:    <= 0.109 -> 1
    - ug/mL: <= 0.7   -> 1
    """
    if pd.isna(ic50_value):
        return np.nan

    try:
        v = float(ic50_value)
    except Exception:
        return np.nan

    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= THR_IC50_UM)
    elif ic50_unit == "ug/mL":
        return int(v <= THR_IC50_UGML)
    else:
        return np.nan


# =========================
# 2) 特征处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    """
    分类列统一清洗：
    - 转字符串
    - 去首尾空格
    - 合并中间多余空格
    - 小写化
    - 空字符串 / nan / none -> 缺失
    """
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    })
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    # 数值列强制转数值
    for c in numeric_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    # 分类列统一清洗
    for c in categorical_cols:
        X[c] = clean_text_series(X[c])

    return X


def base_feature_name(transformed_name: str) -> str:
    """
    把 one-hot / missing indicator 后的特征名，映射回原始特征名
    方便做构效关系分析
    """
    if transformed_name.startswith("num__missingindicator_"):
        return transformed_name.replace("num__missingindicator_", "")
    if transformed_name.startswith("num__"):
        return transformed_name.replace("num__", "")

    if transformed_name.startswith("cat__shape_"):
        return "shape"
    if transformed_name.startswith("cat__surface treatment_"):
        return "surface treatment"
    if transformed_name.startswith("cat__dispersion medium_"):
        return "dispersion medium"
    if transformed_name.startswith("cat__Substrate _"):
        return "Substrate"

    return transformed_name


# =========================
# 3) 主流程
# =========================
def main():
    df = pd.read_excel(SOD_PATH)

    # ---- 解析 IC50
    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    # ---- 打标签
    df["y_IC50Low"] = df.apply(
        lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]),
        axis=1
    )

    # ---- 特征列
    feature_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "shape", "size (nm)", "surface treatment", "dispersion medium",
        "pH", "temperature/oC", "Substrate "
    ]

    numeric_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]

    categorical_cols = [
        "shape", "surface treatment", "dispersion medium", "Substrate "
    ]

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"以下特征列不存在：{missing_cols}")

    # ---- 仅保留有效标签样本
    df_use = df[df["y_IC50Low"].notna()].copy()
    X_raw = prepare_features(df_use, feature_cols, numeric_cols, categorical_cols)
    y = df_use["y_IC50Low"].astype(int).values

    # ---- 预处理（dense，便于 SHAP）
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median", add_indicator=True))
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
                ("onehot", oh),
            ]), categorical_cols),
        ],
        remainder="drop"
    )

    X_trans = preprocess.fit_transform(X_raw)
    feature_names = preprocess.get_feature_names_out()
    X_df = pd.DataFrame(X_trans, columns=feature_names)

    # ---- 最终 XGBoost 模型（按你前面粗筛代码风格）
    model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,   # 本地 SHAP 时建议先用 1，更稳
        verbosity=0
    )

    model.fit(X_trans, y)

    # ---- 基本训练集内参考指标
    p = model.predict_proba(X_trans)[:, 1]
    pred = (p >= 0.5).astype(int)

    metrics = pd.DataFrame([{
        "n_samples": len(y),
        "n_positive": int((y == 1).sum()),
        "n_negative": int((y == 0).sum()),
        "train_auc": float(roc_auc_score(y, p)),
        "train_acc": float(accuracy_score(y, pred)),
        "train_mcc": float(matthews_corrcoef(y, pred)),
        "train_kappa": float(cohen_kappa_score(y, pred)),
        "threshold_uM": THR_IC50_UM,
        "threshold_ugmL": THR_IC50_UGML,
    }])
    metrics.to_csv(OUT_DIR / "sod_xgb_shap_metrics.csv", index=False, encoding="utf-8-sig")

    # =========================
    # 4) SHAP
    # =========================
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_trans)
    shap_values = np.asarray(shap_values)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    # ---- transformed feature 层面 SHAP
    trans_df = pd.DataFrame({
        "transformed_feature": feature_names,
        "base_feature": [base_feature_name(f) for f in feature_names],
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    trans_df.to_csv(
        OUT_DIR / "sod_xgb_transformed_feature_importance_shap.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # ---- 聚合回原始特征层面 SHAP
    agg_shap = (
        trans_df.groupby("base_feature", as_index=False)["mean_abs_shap"]
        .sum()
        .sort_values("mean_abs_shap", ascending=False)
    )

    agg_shap.to_csv(
        OUT_DIR / "sod_xgb_base_feature_importance_shap.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # =========================
    # 5) XGBoost 内置 importance
    # =========================
    booster = model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")
    weight_dict = booster.get_score(importance_type="weight")
    cover_dict = booster.get_score(importance_type="cover")

    xgb_imp = []
    for i, name in enumerate(feature_names):
        key = f"f{i}"
        xgb_imp.append({
            "transformed_feature": name,
            "base_feature": base_feature_name(name),
            "gain": float(gain_dict.get(key, 0.0)),
            "weight": float(weight_dict.get(key, 0.0)),
            "cover": float(cover_dict.get(key, 0.0)),
        })

    xgb_imp = pd.DataFrame(xgb_imp)
    xgb_imp.to_csv(
        OUT_DIR / "sod_xgb_transformed_feature_importance_builtin.csv",
        index=False,
        encoding="utf-8-sig"
    )

    agg_builtin = (
        xgb_imp.groupby("base_feature", as_index=False)[["gain", "weight", "cover"]]
        .sum()
        .sort_values("gain", ascending=False)
    )

    agg_builtin.to_csv(
        OUT_DIR / "sod_xgb_base_feature_importance_builtin.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # ---- 合并一个总表
    summary_merged = (
        agg_shap.merge(agg_builtin, on="base_feature", how="left")
        .sort_values("mean_abs_shap", ascending=False)
    )
    summary_merged.to_csv(
        OUT_DIR / "sod_xgb_feature_importance_summary_merged.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # =========================
    # 6) 画图
    # =========================
    plt.rcParams["axes.unicode_minus"] = False

    # 6.1 SHAP beeswarm（transformed top20）
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_df, show=False, max_display=20)
    plt.title("SOD XGBoost SHAP summary (top 20 transformed features)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "sod_xgb_shap_beeswarm_top20.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 6.2 SHAP bar（transformed top20）
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_df, plot_type="bar", show=False, max_display=20)
    plt.title("SOD XGBoost mean(|SHAP|) bar (top 20 transformed features)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "sod_xgb_shap_bar_top20.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 6.3 聚合后 SHAP bar（base feature）
    top_agg_shap = agg_shap.head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_agg_shap["base_feature"], top_agg_shap["mean_abs_shap"])
    plt.xlabel("sum of mean(|SHAP|)")
    plt.title("SOD XGBoost aggregated SHAP importance (top 20 base features)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "sod_xgb_shap_base_feature_bar_top20.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 6.4 XGBoost gain（base feature）
    top_gain = agg_builtin.sort_values("gain", ascending=False).head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_gain["base_feature"], top_gain["gain"])
    plt.xlabel("gain")
    plt.title("SOD XGBoost built-in importance by gain (top 20 base features)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "sod_xgb_builtin_gain_top20.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 6.5 XGBoost split count（base feature）
    top_weight = agg_builtin.sort_values("weight", ascending=False).head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_weight["base_feature"], top_weight["weight"])
    plt.xlabel("split count")
    plt.title("SOD XGBoost built-in importance by split count (top 20 base features)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "sod_xgb_builtin_weight_top20.png", dpi=300, bbox_inches="tight")
    plt.close()

    # =========================
    # 7) 保存模型
    # =========================
    joblib.dump(
        {
            "preprocess": preprocess,
            "model": model,
            "feature_names": feature_names,
            "threshold_uM": THR_IC50_UM,
            "threshold_ugmL": THR_IC50_UGML,
        },
        OUT_DIR / "sod_xgb_0p7_model_for_shap.joblib"
    )

    print("=" * 60)
    print("Done.")
    print("=" * 60)
    print("Output folder:")
    print(OUT_DIR)
    print("\nMetrics:")
    print(metrics.to_string(index=False))
    print("\nTop 15 aggregated SHAP features:")
    print(agg_shap.head(15).to_string(index=False))
    print("\nTop 15 aggregated XGBoost gain features:")
    print(agg_builtin.head(15).to_string(index=False))


if __name__ == "__main__":
    main()

Done.
Output folder:
C:\Users\86158\Desktop\sod_xgb_shap_0p7

Metrics:
 n_samples  n_positive  n_negative  train_auc  train_acc  train_mcc  train_kappa  threshold_uM  threshold_ugmL
        50          24          26   0.998397       0.96     0.9226     0.919614         0.109             0.7

Top 15 aggregated SHAP features:
          base_feature  mean_abs_shap
    minor metal number       0.996565
     main metal number       0.663421
             size (nm)       0.654728
             contain N       0.645356
main  metal proportion       0.624692
                    pH       0.485442
                 shape       0.365682
    main metal valence       0.347882
        temperature/oC       0.251783
     dispersion medium       0.210811
             Substrate       0.181618
   minor metal valence       0.169232
             contain S       0.127925
minor metal percentage       0.053553
     surface treatment       0.044014

Top 15 aggregated XGBoost gain features:
          base_feature 


uM regression
样本数: 27
唯一 group 数: 12
模型数: 13

[Running] uM | LinearRegression ...

[Running] uM | Ridge ...

[Running] uM | Lasso ...

[Running] uM | ElasticNet ...

[Running] uM | SVR_RBF ...

[Running] uM | KNN ...

[Running] uM | DecisionTree ...

[Running] uM | RF ...

[Running] uM | ExtraTrees ...

[Running] uM | GradientBoosting ...

[Running] uM | HistGB ...

[Running] uM | AdaBoost ...

[Running] uM | XGBoost ...

Summary:
unit            model     r2_mean   r2_median     r2_std     r2_max  mae_mean  rmse_mean  n_splits
  uM          SVR_RBF   -0.259817    0.035603   0.902413   0.939514  0.969534   1.546024        16
  uM           HistGB   -0.566144   -0.427767   0.574327  -0.003350  1.130642   1.643667        16
  uM         AdaBoost   -3.516410   -1.122338   5.625018  -0.143985  1.581218   2.037296        16
  uM              KNN   -5.461728   -3.447219   5.012253  -0.544811  1.858481   2.378045        16
  uM               RF   -7.147571   -4.130664   7.329884  -0.607435  


Top 10:
unit      scheme   model  n_samples  n_groups  n_folds  r2_log_oof  mae_log_oof  rmse_log_oof  pearson_r_oof  pearson_r2_oof
  uM full_svd_10 SVR_RBF         27        12        5   -0.030870     1.665781      2.879304       0.115351        0.013306
  uM      no_cat SVR_RBF         27        12        5   -0.053984     1.333392      2.911404       0.129282        0.016714
  uM        full  HistGB         27        12        5   -0.054613     1.709281      2.912273      -0.319448        0.102047
  uM      no_cat  HistGB         27        12        5   -0.054613     1.709281      2.912273      -0.319448        0.102047
  uM  paper_like  HistGB         27        12        5   -0.054613     1.709281      2.912273      -0.319448        0.102047
  uM full_svd_20 SVR_RBF         27        12        5   -0.067646     1.669468      2.930213      -0.308564        0.095212
  uM full_svd_30 SVR_RBF         27        12        5   -0.072271     1.650110      2.936553      -0.310205        

In [5]:
# -*- coding: utf-8 -*-
"""
SOD 回归进一步优化版（按 DOI 外推）
方法：
1) 稀有类别合并
2) 低方差特征删除
3) SelectKBest(f_regression) 监督筛选
4) SVD 降维
5) Ridge / ElasticNet / SVR / KernelRidge / Huber / BayesianRidge
6) GroupKFold OOF 评估

输出：
- uM_refined2_summary.csv
- ugmL_refined2_summary.csv
- uM_refined2_oof_predictions.csv
- ugmL_refined2_oof_predictions.csv
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression

from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# 0) 路径与配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_regression_doi_refined2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IC50 = "IC50/μM"
COL_DOI = "data reference doi"

MAX_FOLDS = 5
RANDOM_SEED = 42

MIN_FREQ_CAT_OPTIONS = [2, 3]
VAR_THRESHOLDS = [0.0, 1e-5, 1e-4]
K_BEST_OPTIONS = [None, 8, 10, 12, 15]
SVD_DIMS = [None, 6, 8, 10, 12]

OUT_USED_DATA = OUT_DIR / "used_data.csv"
OUT_FEATURE_INFO = OUT_DIR / "feature_info.csv"


# =========================
# 1) IC50 解析
# =========================
def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================
# 2) 基础预处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=2, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )


# =========================
# 3) 指标
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def evaluate_oof(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


# =========================
# 4) 模型工厂
# =========================
def get_candidate_models():
    models = []

    for alpha in [0.3, 1, 3, 10, 30, 100]:
        models.append(("Ridge", {"alpha": alpha}))

    for alpha in [0.001, 0.01, 0.03, 0.1]:
        for l1_ratio in [0.1, 0.3, 0.5, 0.7]:
            models.append(("ElasticNet", {"alpha": alpha, "l1_ratio": l1_ratio}))

    models.append(("BayesianRidge", {}))

    for alpha in [0.0001, 0.001, 0.01]:
        for epsilon in [1.1, 1.35, 1.5]:
            models.append(("Huber", {"alpha": alpha, "epsilon": epsilon}))

    for C in [3, 10, 30]:
        for gamma in ["scale", 0.01, 0.03, 0.1, 0.3]:
            for epsilon in [0.05, 0.1, 0.2]:
                models.append(("SVR_RBF", {"C": C, "gamma": gamma, "epsilon": epsilon}))

    for alpha in [0.01, 0.1, 1, 10]:
        for gamma in [0.01, 0.03, 0.1, 0.3]:
            models.append(("KernelRidge_RBF", {"alpha": alpha, "gamma": gamma}))

    return models


def instantiate_model(model_name, params):
    if model_name == "Ridge":
        return Ridge(random_state=RANDOM_SEED, **params)
    if model_name == "ElasticNet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_SEED, **params)
    if model_name == "BayesianRidge":
        return BayesianRidge(**params)
    if model_name == "Huber":
        return HuberRegressor(**params)
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def model_needs_scaling(model_name):
    return model_name in {
        "Ridge", "ElasticNet", "BayesianRidge", "Huber", "SVR_RBF", "KernelRidge_RBF"
    }


# =========================
# 5) GroupKFold OOF
# =========================
def group_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for fold, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        pred = np.asarray(pred).reshape(-1)
        oof_pred[te] = pred

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 6) 单位主程序
# =========================
def run_unit(unit_name, df_unit, feature_cols, numeric_cols, categorical_cols):
    df_unit = df_unit.copy()
    df_unit = df_unit[df_unit["IC50_value"].notna() & (df_unit["IC50_value"] > 0)].copy()

    if len(df_unit) < 12:
        print(f"[Skip] {unit_name}: 样本太少")
        return

    y_log = np.log10(df_unit["IC50_value"].astype(float).values)

    groups = df_unit[COL_DOI].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    groups = groups.values

    X_full = prepare_features(df_unit, feature_cols, numeric_cols, categorical_cols)

    results = []
    oof_save = []

    candidate_models = get_candidate_models()

    for min_freq_cat in MIN_FREQ_CAT_OPTIONS:
        for var_thr in VAR_THRESHOLDS:
            for k_best in K_BEST_OPTIONS:
                for svd_dim in SVD_DIMS:
                    # k_best 之后再 SVD，没必要 svd_dim > k_best
                    if (k_best is not None) and (svd_dim is not None) and (svd_dim >= k_best):
                        continue

                    for model_name, params in candidate_models:
                        model = instantiate_model(model_name, params)

                        preprocess = build_preprocess(
                            numeric_cols=numeric_cols,
                            categorical_cols=categorical_cols,
                            min_freq_cat=min_freq_cat,
                            sparse=True
                        )

                        def builder():
                            steps = [("preprocess", preprocess)]

                            # 低方差过滤
                            steps.append(("var", VarianceThreshold(threshold=var_thr)))

                            # 监督筛选
                            if k_best is not None:
                                steps.append(("kbest", SelectKBest(score_func=f_regression, k=k_best)))

                            # SVD
                            if svd_dim is not None:
                                steps.append(("svd", TruncatedSVD(n_components=svd_dim, random_state=RANDOM_SEED)))

                            # 标准化
                            if model_needs_scaling(model_name):
                                steps.append(("scaler", StandardScaler(with_mean=True)))

                            steps.append(("model", model))
                            return Pipeline(steps)

                        out = group_oof_predict(X_full, y_log, groups, builder)
                        if out is None:
                            continue

                        pred, ok, n_folds = out
                        metrics = evaluate_oof(y_log[ok], pred[ok])

                        scheme_parts = [f"minfreq{min_freq_cat}", f"var{var_thr:g}"]
                        scheme_parts.append(f"k{k_best}" if k_best is not None else "kNone")
                        scheme_parts.append(f"svd{svd_dim}" if svd_dim is not None else "svdNone")
                        scheme = "_".join(scheme_parts)

                        results.append({
                            "unit": unit_name,
                            "scheme": scheme,
                            "model": model_name,
                            "params": str(params),
                            "n_samples": len(df_unit),
                            "n_groups": len(np.unique(groups)),
                            "n_folds": n_folds,
                            **metrics
                        })

                        oof_save.append(pd.DataFrame({
                            "unit": unit_name,
                            "scheme": scheme,
                            "model": model_name,
                            "params": str(params),
                            "index": df_unit.index.values,
                            "y_log_true": y_log,
                            "y_log_oof_pred": pred
                        }))

    res = pd.DataFrame(results).sort_values(
        by=["r2_log_oof", "pearson_r2_oof", "mae_log_oof"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    res.to_csv(OUT_DIR / f"{unit_name}_refined2_summary.csv", index=False, encoding="utf-8-sig")

    if oof_save:
        pd.concat(oof_save, ignore_index=True).to_csv(
            OUT_DIR / f"{unit_name}_refined2_oof_predictions.csv",
            index=False, encoding="utf-8-sig"
        )

    print("\n" + "=" * 60)
    print(f"{unit_name} Top 20")
    print("=" * 60)
    print(res.head(20).to_string(index=False))


# =========================
# 7) 主程序
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)
    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    df_use = df[
        df["IC50_unit"].isin(["uM", "ug/mL"]) &
        df["IC50_value"].notna() &
        (pd.to_numeric(df["IC50_value"], errors="coerce") > 0)
    ].copy()

    df_use.to_csv(OUT_USED_DATA, index=False, encoding="utf-8-sig")

    feature_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "shape", "size (nm)", "surface treatment", "dispersion medium",
        "pH", "temperature/oC", "Substrate "
    ]

    numeric_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]

    categorical_cols = ["shape", "surface treatment", "dispersion medium", "Substrate "]

    feature_info = pd.DataFrame({
        "feature": feature_cols,
        "type": ["numeric" if c in numeric_cols else "categorical" for c in feature_cols]
    })
    feature_info.to_csv(OUT_FEATURE_INFO, index=False, encoding="utf-8-sig")

    run_unit("uM", df_use[df_use["IC50_unit"] == "uM"].copy(),
             feature_cols, numeric_cols, categorical_cols)
    run_unit("ugmL", df_use[df_use["IC50_unit"] == "ug/mL"].copy(),
             feature_cols, numeric_cols, categorical_cols)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


uM Top 20
unit               scheme   model                                       params  n_samples  n_groups  n_folds  r2_log_oof  mae_log_oof  rmse_log_oof  pearson_r_oof  pearson_r2_oof
  uM     var0_kNone_svd12 SVR_RBF      {'C': 30, 'gamma': 0.1, 'epsilon': 0.1}         27        12        5    0.014244     1.567797      2.815596       0.186002        0.034597
  uM var1e-05_kNone_svd12 SVR_RBF      {'C': 30, 'gamma': 0.1, 'epsilon': 0.1}         27        12        5    0.014244     1.567797      2.815596       0.186002        0.034597
  uM     var0_kNone_svd12 SVR_RBF      {'C': 30, 'gamma': 0.1, 'epsilon': 0.2}         27        12        5    0.013740     1.567947      2.816315       0.182510        0.033310
  uM var1e-05_kNone_svd12 SVR_RBF      {'C': 30, 'gamma': 0.1, 'epsilon': 0.2}         27        12        5    0.013740     1.567947      2.816315       0.182510        0.033310
  uM     var0_kNone_svd12 SVR_RBF     {'C': 30, 'gamma': 0.1, 'epsilon': 0.05}         27     

In [6]:
# -*- coding: utf-8 -*-
"""
SOD 回归：两单位分开（uM / ugmL）+ DOI 分组外推 + OOF 评估 + 自动选最优 + 保存最终模型

最终模型结构：
preprocess (num+cat+rare_grouping+onehot) -> VarianceThreshold -> SVD -> StandardScaler -> SVR_RBF

输出目录：
C:\\Users\\86158\\Desktop\\sod_reg_best_models

输出文件（每单位一套）：
- <unit>_search_summary.csv
- <unit>_oof_predictions.csv
- <unit>_best_config.json
- <unit>_best_model_refit_full.joblib
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# 0) 路径与配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_reg_best_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IC50 = "IC50/μM"
COL_DOI = "data reference doi"

RANDOM_SEED = 42
MAX_FOLDS = 5

# 你已验证：min_freq=3 最稳
MIN_FREQ_CAT = 3

# 这两个你之前基本等效，保留是为了“确认不是偶然”
VAR_THRESHOLDS = [0.0, 1e-5]

# ---- 围绕你跑出来的最优附近做“小范围搜索” ----
# uM：你最优在 svd12 + gamma=0.1 + C=30 + eps=0.1
UM_SVD_DIMS = [10, 12, 14]
UM_C = [10, 30, 50]
UM_GAMMA = ["scale", 0.03, 0.1, 0.3]
UM_EPS = [0.05, 0.1, 0.2]

# ugmL：你最优在 svd10 + gamma=0.01 + C=30 + eps=0.2
UG_SVD_DIMS = [8, 10, 12]
UG_C = [10, 30, 50]
UG_GAMMA = [0.003, 0.01, 0.03, 0.1]
UG_EPS = [0.05, 0.1, 0.2]


# =========================
# 1) IC50 解析
# =========================
def parse_ic50_cell(x):
    """
    解析 IC50：
    - 纯数字 -> 默认 uM
    - '数值 + uM/μM/µM' -> uM
    - '数值 + ug/mL/μg/mL/µg/mL' -> ug/mL
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================
# 2) 特征处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补 0（和你前面一致）
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """
    训练集内：出现频次 < min_freq 的类别合并为 __other__
    """
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=MIN_FREQ_CAT)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )


# =========================
# 3) 评估：GroupKFold OOF
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r*r) if not pd.isna(r) else np.nan,
    }


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for fold, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        oof_pred[te] = np.asarray(pred).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 4) 单位搜索：自动选最优
# =========================
def search_best_for_unit(unit_name, df_unit, X_raw, y_log, groups, preprocess,
                         var_thresholds, svd_dims, C_list, gamma_list, eps_list):

    rows = []
    oof_store = []

    for var_thr in var_thresholds:
        for svd_dim in svd_dims:
            for C in C_list:
                for gamma in gamma_list:
                    for eps in eps_list:

                        def builder():
                            return Pipeline([
                                ("preprocess", preprocess),
                                ("var", VarianceThreshold(threshold=var_thr)),
                                ("svd", TruncatedSVD(n_components=svd_dim, random_state=RANDOM_SEED)),
                                ("scaler", StandardScaler(with_mean=True)),
                                ("model", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=eps)),
                            ])

                        out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                        if out is None:
                            continue

                        pred, ok, n_folds = out
                        m = eval_metrics(y_log[ok], pred[ok])

                        rows.append({
                            "unit": unit_name,
                            "var_thr": var_thr,
                            "svd_dim": svd_dim,
                            "C": C,
                            "gamma": gamma,
                            "epsilon": eps,
                            "n_samples": len(df_unit),
                            "n_groups": int(len(np.unique(groups))),
                            "n_folds": int(n_folds),
                            **m
                        })

                        # 只保存少量顶级结果的 OOF（避免文件太大）
                        oof_store.append((m["r2_log_oof"], {
                            "var_thr": var_thr, "svd_dim": svd_dim, "C": C, "gamma": gamma, "epsilon": eps
                        }, pred))

    res = pd.DataFrame(rows)
    res = res.sort_values(by=["r2_log_oof", "rmse_log_oof", "mae_log_oof"],
                          ascending=[False, True, True]).reset_index(drop=True)

    best = res.iloc[0].to_dict()

    # 保存 summary
    res.to_csv(OUT_DIR / f"{unit_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    # 保存 best config
    best_config = {
        "unit": unit_name,
        "best_params": {
            "var_thr": float(best["var_thr"]),
            "svd_dim": int(best["svd_dim"]),
            "C": float(best["C"]),
            "gamma": best["gamma"],
            "epsilon": float(best["epsilon"]),
        },
        "metrics": {
            "r2_log_oof": float(best["r2_log_oof"]),
            "mae_log_oof": float(best["mae_log_oof"]),
            "rmse_log_oof": float(best["rmse_log_oof"]),
            "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
            "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
        },
        "fixed_choices": {
            "target": "log10(IC50)",
            "group_split": "GroupKFold by DOI",
            "min_freq_cat": MIN_FREQ_CAT,
            "model_family": "SVR_RBF",
        }
    }
    with open(OUT_DIR / f"{unit_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    # 生成 best 的 OOF 预测文件（按 best config 重新跑一遍，确保一致）
    bp = best_config["best_params"]
    def best_builder():
        return Pipeline([
            ("preprocess", preprocess),
            ("var", VarianceThreshold(threshold=bp["var_thr"])),
            ("svd", TruncatedSVD(n_components=bp["svd_dim"], random_state=RANDOM_SEED)),
            ("scaler", StandardScaler(with_mean=True)),
            ("model", SVR(kernel="rbf", C=bp["C"], gamma=bp["gamma"], epsilon=bp["epsilon"])),
        ])

    pred, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    oof_df = pd.DataFrame({
        "index": df_unit.index.values,
        "y_log_true": y_log,
        "y_log_oof_pred": pred
    })
    oof_df.to_csv(OUT_DIR / f"{unit_name}_oof_predictions.csv", index=False, encoding="utf-8-sig")

    # 全量重训保存模型
    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    joblib.dump({
        "unit": unit_name,
        "pipeline": final_pipe,
        "feature_cols": FEATURE_COLS,
        "numeric_cols": NUMERIC_COLS,
        "categorical_cols": CATEGORICAL_COLS,
        "target": "log10(IC50)",
        "best_config": best_config
    }, OUT_DIR / f"{unit_name}_best_model_refit_full.joblib")

    return best_config


# =========================
# 5) 主程序
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate "]


def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    # 有效值
    df_use = df[
        df["IC50_unit"].isin(["uM", "ug/mL"]) &
        df["IC50_value"].notna() &
        (pd.to_numeric(df["IC50_value"], errors="coerce") > 0)
    ].copy()

    if df_use.empty:
        raise RuntimeError("没有可用于回归的有效样本。")

    # 分单位
    for unit_name in ["uM", "ug/mL"]:
        df_unit = df_use[df_use["IC50_unit"] == unit_name].copy()
        if len(df_unit) < 10:
            print(f"[Skip] {unit_name}: 样本太少 {len(df_unit)}")
            continue

        # 特征 + 目标
        X_raw = prepare_features(df_unit, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS)
        y_log = np.log10(df_unit["IC50_value"].astype(float).values)

        # DOI 分组（缺失 DOI 的行给唯一伪 DOI）
        groups = df_unit[COL_DOI].fillna("").astype(str).str.strip()
        miss = groups.eq("")
        groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
        groups = groups.values

        preprocess = build_preprocess(NUMERIC_COLS, CATEGORICAL_COLS, sparse=True)

        if unit_name == "uM":
            best = search_best_for_unit(
                unit_name="uM",
                df_unit=df_unit,
                X_raw=X_raw,
                y_log=y_log,
                groups=groups,
                preprocess=preprocess,
                var_thresholds=VAR_THRESHOLDS,
                svd_dims=UM_SVD_DIMS,
                C_list=UM_C,
                gamma_list=UM_GAMMA,
                eps_list=UM_EPS
            )
        else:
            best = search_best_for_unit(
                unit_name="ugmL",
                df_unit=df_unit,
                X_raw=X_raw,
                y_log=y_log,
                groups=groups,
                preprocess=preprocess,
                var_thresholds=VAR_THRESHOLDS,
                svd_dims=UG_SVD_DIMS,
                C_list=UG_C,
                gamma_list=UG_GAMMA,
                eps_list=UG_EPS
            )

        print("\n" + "=" * 70)
        print(f"[BEST] {best['unit']}")
        print(best["best_params"])
        print(best["metrics"])
        print("=" * 70)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


[BEST] uM
{'var_thr': 0.0, 'svd_dim': 12, 'C': 50.0, 'gamma': 0.1, 'epsilon': 0.2}
{'r2_log_oof': 0.05854908903371547, 'mae_log_oof': 1.5645999265904043, 'rmse_log_oof': 2.751594238867906, 'pearson_r_oof': 0.2552794103981822, 'pearson_r2_oof': 0.06516757737324354}

[BEST] ugmL
{'var_thr': 0.0, 'svd_dim': 10, 'C': 50.0, 'gamma': 0.01, 'epsilon': 0.2}
{'r2_log_oof': 0.6053020766777997, 'mae_log_oof': 0.5974514130293919, 'rmse_log_oof': 0.7547316385256279, 'pearson_r_oof': 0.8048334192841361, 'pearson_r2_oof': 0.647756832796594}

Saved to: C:\Users\86158\Desktop\sod_reg_best_models


In [19]:
# -*- coding: utf-8 -*-
"""
SOD 回归：Km / Vmax（同 IC50 版本逻辑：DOI 分组外推 + OOF + 自动选最优 + 保存最终模型）

最终模型结构：
preprocess (num+cat+rare_grouping+onehot) -> VarianceThreshold -> SVD -> StandardScaler -> SVR_RBF

输出目录：
C:\\Users\\86158\\Desktop\\sod_reg_best_models_km_vmax

输出文件（每个 target 一套）：
- <target>_search_summary.csv
- <target>_oof_predictions.csv
- <target>_best_config.json
- <target>_best_model_refit_full.joblib
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# 0) 路径与配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

# 为避免覆盖你之前 IC50 的输出，我这里换了一个新目录；如需保持原样可改回去
OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_reg_best_models_km_vmax")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"

# 目标列（按你之前与表格一致的命名）
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5

# 你已验证：min_freq=3 最稳
MIN_FREQ_CAT = 3

# 这两个你之前基本等效，保留是为了“确认不是偶然”
VAR_THRESHOLDS = [0.0, 1e-5]

# ---- 围绕你之前找到的 SVR+SVD 最优附近做“小范围搜索” ----
# 这里保留两套 grid：你可以理解为 “Km 用一套，Vmax 用一套”
# （完全沿用你原来 uM / ugmL 两套网格的结构）

# Grid A（原 uM 网格）
GRID_A_SVD_DIMS = [10, 12, 14]
GRID_A_C = [10, 30, 50]
GRID_A_GAMMA = ["scale", 0.03, 0.1, 0.3]
GRID_A_EPS = [0.05, 0.1, 0.2]

# Grid B（原 ugmL 网格）
GRID_B_SVD_DIMS = [8, 10, 12]
GRID_B_C = [10, 30, 50]
GRID_B_GAMMA = [0.003, 0.01, 0.03, 0.1]
GRID_B_EPS = [0.05, 0.1, 0.2]


# =========================
# 1) 特征处理（保持不变）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补 0（和你前面一致）
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """
    训练集内：出现频次 < min_freq 的类别合并为 __other__
    """
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=MIN_FREQ_CAT)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )


# =========================
# 2) 评估：GroupKFold OOF（保持不变）
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r*r) if not pd.isna(r) else np.nan,
    }


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for fold, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        oof_pred[te] = np.asarray(pred).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 3) 搜索：自动选最优（保持不变，只把 unit_name -> target_name）
# =========================
def search_best_for_target(target_name, df_unit, X_raw, y_log, groups, preprocess,
                           var_thresholds, svd_dims, C_list, gamma_list, eps_list):

    rows = []

    for var_thr in var_thresholds:
        for svd_dim in svd_dims:
            for C in C_list:
                for gamma in gamma_list:
                    for eps in eps_list:

                        def builder():
                            return Pipeline([
                                ("preprocess", preprocess),
                                ("var", VarianceThreshold(threshold=var_thr)),
                                ("svd", TruncatedSVD(n_components=svd_dim, random_state=RANDOM_SEED)),
                                ("scaler", StandardScaler(with_mean=True)),
                                ("model", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=eps)),
                            ])

                        out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                        if out is None:
                            continue

                        pred, ok, n_folds = out
                        m = eval_metrics(y_log[ok], pred[ok])

                        rows.append({
                            "target": target_name,
                            "var_thr": float(var_thr),
                            "svd_dim": int(svd_dim),
                            "C": float(C),
                            "gamma": gamma,
                            "epsilon": float(eps),
                            "n_samples": int(len(df_unit)),
                            "n_groups": int(len(np.unique(groups))),
                            "n_folds": int(n_folds),
                            **m
                        })

    res = pd.DataFrame(rows)
    if res.empty:
        raise RuntimeError(f"{target_name}: 所有候选参数训练/评估都失败了。")

    res = res.sort_values(by=["r2_log_oof", "rmse_log_oof", "mae_log_oof"],
                          ascending=[False, True, True]).reset_index(drop=True)

    best = res.iloc[0].to_dict()

    # 保存 summary
    res.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    # 保存 best config
    best_config = {
        "target": target_name,
        "best_params": {
            "var_thr": float(best["var_thr"]),
            "svd_dim": int(best["svd_dim"]),
            "C": float(best["C"]),
            "gamma": best["gamma"],
            "epsilon": float(best["epsilon"]),
        },
        "metrics": {
            "r2_log_oof": float(best["r2_log_oof"]),
            "mae_log_oof": float(best["mae_log_oof"]),
            "rmse_log_oof": float(best["rmse_log_oof"]),
            "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
            "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
        },
        "fixed_choices": {
            "target_space": "log10(target_value)",
            "group_split": "GroupKFold by DOI",
            "min_freq_cat": MIN_FREQ_CAT,
            "model_family": "SVR_RBF",
        }
    }
    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    # 生成 best 的 OOF 预测文件（按 best config 重新跑一遍，确保一致）
    bp = best_config["best_params"]

    def best_builder():
        return Pipeline([
            ("preprocess", preprocess),
            ("var", VarianceThreshold(threshold=bp["var_thr"])),
            ("svd", TruncatedSVD(n_components=bp["svd_dim"], random_state=RANDOM_SEED)),
            ("scaler", StandardScaler(with_mean=True)),
            ("model", SVR(kernel="rbf", C=bp["C"], gamma=bp["gamma"], epsilon=bp["epsilon"])),
        ])

    pred, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    oof_df = pd.DataFrame({
        "index": df_unit.index.values,
        "y_log_true": y_log,
        "y_log_oof_pred": pred
    })
    oof_df.to_csv(OUT_DIR / f"{target_name}_oof_predictions.csv", index=False, encoding="utf-8-sig")

    # 全量重训保存模型
    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    joblib.dump({
        "target": target_name,
        "pipeline": final_pipe,
        "feature_cols": FEATURE_COLS,
        "numeric_cols": NUMERIC_COLS,
        "categorical_cols": CATEGORICAL_COLS,
        "target_space": "log10(target_value)",
        "best_config": best_config
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    return best_config


# =========================
# 4) 主程序：Km + Vmax（其余不变）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate "]


def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    # 简单列检查（避免列名不一致导致 silent failure）
    required_cols = set([COL_DOI, COL_KM, COL_VMAX] + FEATURE_COLS)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"以下列在 SOD 表中不存在：{missing}\n"
            f"请核对 Km/Vmax 列名是否为：'{COL_KM}' 和 '{COL_VMAX}'"
        )

    preprocess = build_preprocess(NUMERIC_COLS, CATEGORICAL_COLS, sparse=True)

    # ===== Target 1: Km =====
    df_km = df[pd.to_numeric(df[COL_KM], errors="coerce").notna() & (pd.to_numeric(df[COL_KM], errors="coerce") > 0)].copy()
    if len(df_km) < 10:
        print(f"[Skip] Km: 样本太少 {len(df_km)}")
    else:
        X_raw = prepare_features(df_km, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS)
        y_log = np.log10(pd.to_numeric(df_km[COL_KM], errors="coerce").astype(float).values)

        groups = df_km[COL_DOI].fillna("").astype(str).str.strip()
        miss = groups.eq("")
        groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_km.index[miss]]
        groups = groups.values

        best = search_best_for_target(
            target_name="Km",
            df_unit=df_km,
            X_raw=X_raw,
            y_log=y_log,
            groups=groups,
            preprocess=preprocess,
            var_thresholds=VAR_THRESHOLDS,
            svd_dims=GRID_A_SVD_DIMS,
            C_list=GRID_A_C,
            gamma_list=GRID_A_GAMMA,
            eps_list=GRID_A_EPS
        )
        print("\n" + "=" * 70)
        print("[BEST] Km")
        print(best["best_params"])
        print(best["metrics"])
        print("=" * 70)

    # ===== Target 2: Vmax =====
    df_v = df[pd.to_numeric(df[COL_VMAX], errors="coerce").notna() & (pd.to_numeric(df[COL_VMAX], errors="coerce") > 0)].copy()
    if len(df_v) < 10:
        print(f"[Skip] Vmax: 样本太少 {len(df_v)}")
    else:
        X_raw = prepare_features(df_v, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS)
        y_log = np.log10(pd.to_numeric(df_v[COL_VMAX], errors="coerce").astype(float).values)

        groups = df_v[COL_DOI].fillna("").astype(str).str.strip()
        miss = groups.eq("")
        groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_v.index[miss]]
        groups = groups.values

        best = search_best_for_target(
            target_name="Vmax",
            df_unit=df_v,
            X_raw=X_raw,
            y_log=y_log,
            groups=groups,
            preprocess=preprocess,
            var_thresholds=VAR_THRESHOLDS,
            svd_dims=GRID_B_SVD_DIMS,
            C_list=GRID_B_C,
            gamma_list=GRID_B_GAMMA,
            eps_list=GRID_B_EPS
        )
        print("\n" + "=" * 70)
        print("[BEST] Vmax")
        print(best["best_params"])
        print(best["metrics"])
        print("=" * 70)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


[BEST] Km
{'var_thr': 0.0, 'svd_dim': 10, 'C': 10.0, 'gamma': 0.03, 'epsilon': 0.2}
{'r2_log_oof': -0.01261184774369939, 'mae_log_oof': 1.385070366740806, 'rmse_log_oof': 1.639565177019436, 'pearson_r_oof': 0.3160134005471548, 'pearson_r2_oof': 0.09986446932537651}

[BEST] Vmax
{'var_thr': 0.0, 'svd_dim': 8, 'C': 30.0, 'gamma': 0.003, 'epsilon': 0.05}
{'r2_log_oof': 0.25957222552792547, 'mae_log_oof': 1.9246323987796938, 'rmse_log_oof': 2.5024926420370326, 'pearson_r_oof': 0.5374079215354008, 'pearson_r2_oof': 0.28880727412899954}

Saved to: C:\Users\86158\Desktop\sod_reg_best_models_km_vmax


In [2]:
# -*- coding: utf-8 -*-
"""
SOD Km / Vmax 粗筛分类模型训练（对齐 IC50 处理细节 + DOI分组切分 + 12模型比选）
改动：
- 不固定尝试 50 次 split
- 改为持续随机生成 GroupShuffleSplit，直到收集到 40~50 个有效 split
- 其余逻辑尽量与原 IC50 风格一致

标签：
- KmLow:   log10(Km)   <= median -> 1
- VmaxHigh:log10(Vmax) >= median -> 1
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_cls_km_vmax_grouped_search_valid40_50")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

# 分组切分设置（保持原来逻辑）
TEST_SIZE = 0.2
SPLIT_SEED = 42
MIN_TEST_SAMPLES = 6
MIN_CLASS_COUNT = 2

# 目标：尽量收集 40~50 个有效 split
TARGET_VALID_SPLITS = 40
MAX_VALID_SPLITS = 50
MAX_TRIES = 10000  # 最多尝试次数，防止死循环


# =========================
# 1) 特征处理（尽量对齐 IC50）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    })
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    # 数值列强制转数值
    for c in numeric_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 对齐你原 IC50 逻辑
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess(numeric_cols, categorical_cols):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )
    return preprocess


# =========================
# 2) 模型定义（对齐 IC50）
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ),

        "LinearSVC": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42
        ),

        "SVC_RBF": SVC(
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),

        "GaussianNB": GaussianNB(),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),

        "RF": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

        "HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )

    return models


def needs_scaling(model_name):
    return model_name in {
        "LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"
    }


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]

    if hasattr(pipe, "decision_function"):
        z = pipe.decision_function(X)
        z = np.asarray(z, dtype=float)
        return 1.0 / (1.0 + np.exp(-z))

    pred = pipe.predict(X)
    return np.asarray(pred, dtype=float)


# =========================
# 3) 标签
# =========================
def make_label_median_log10(values: pd.Series, mode: str):
    """
    mode:
      - low_good:  log10(x) <= median -> 1
      - high_good: log10(x) >= median -> 1
    """
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    v = v.loc[m].astype(float).values
    logv = np.log10(v)
    thr = float(np.median(logv))

    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)

    return m, y, thr


# =========================
# 4) 收集 40~50 个有效 split
# =========================
def collect_valid_group_splits(
    X, y, groups,
    test_size=0.2,
    target_valid_splits=40,
    max_valid_splits=50,
    min_test_samples=6,
    min_class_count=2,
    max_tries=10000,
    base_seed=42
):
    """
    持续随机生成 GroupShuffleSplit，直到收集到 target_valid_splits ~ max_valid_splits 个有效 split。
    若数据本身合法 split 不够，最终返回的数量可能少于 target_valid_splits。
    """
    valid_splits = []
    seen = set()

    tries = 0
    while tries < max_tries and len(valid_splits) < max_valid_splits:
        tries += 1

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=base_seed + tries
        )

        for tr, te in splitter.split(X, y, groups=groups):
            # 用测试集索引去重
            key = tuple(sorted(te.tolist()))
            if key in seen:
                continue
            seen.add(key)

            if len(te) < min_test_samples:
                continue

            ytr, yte = y[tr], y[te]

            if (np.sum(ytr == 0) < min_class_count) or (np.sum(ytr == 1) < min_class_count):
                continue
            if (np.sum(yte == 0) < min_class_count) or (np.sum(yte == 1) < min_class_count):
                continue

            valid_splits.append((tr, te))

            if len(valid_splits) >= max_valid_splits:
                break

    return valid_splits, tries


# =========================
# 5) 单任务训练
# =========================
def run_one_task(task_name, df, y_mask, y, groups, X_all, preprocess, out_prefix):
    df_use = df.loc[y_mask].copy()
    X = X_all.loc[y_mask].copy()

    # 保存 labeled data
    df_use["y"] = y
    out_labeled = OUT_DIR / f"{out_prefix}_labeled_data.csv"
    df_use.to_csv(out_labeled, index=False, encoding="utf-8-sig")

    # 收集有效 split
    valid_split_list, tries = collect_valid_group_splits(
        X=X,
        y=y,
        groups=groups,
        test_size=TEST_SIZE,
        target_valid_splits=TARGET_VALID_SPLITS,
        max_valid_splits=MAX_VALID_SPLITS,
        min_test_samples=MIN_TEST_SAMPLES,
        min_class_count=MIN_CLASS_COUNT,
        max_tries=MAX_TRIES,
        base_seed=SPLIT_SEED
    )

    print("\n" + "=" * 70)
    print(f"[Task] {task_name}")
    print("=" * 70)
    print(f"样本数: {len(y)}")
    print(f"唯一 DOI 组数: {len(np.unique(groups))}")
    print(f"尝试分割次数: {tries}")
    print(f"收集到的有效 split 数: {len(valid_split_list)}")

    if len(valid_split_list) == 0:
        raise RuntimeError(f"{task_name}: 一个有效 split 都没收集到。")

    if len(valid_split_list) < TARGET_VALID_SPLITS:
        print(f"[Warn] 未达到目标 {TARGET_VALID_SPLITS} 个有效 split，仅得到 {len(valid_split_list)} 个。")

    models = get_models()
    rows = []

    for model_name, model in models.items():
        print(f"\n[Running] {task_name} | {model_name} ...")

        for split_id, (tr, te) in enumerate(valid_split_list, start=1):
            ytr, yte = y[tr], y[te]
            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(Xtr, ytr)

                pte = score_to_prob(pipe, Xte)
                pred_te = (pte >= 0.5).astype(int)

                auc = roc_auc_score(yte, pte)
                kappa = cohen_kappa_score(yte, pred_te)
                mcc = matthews_corrcoef(yte, pred_te)
                acc = accuracy_score(yte, pred_te)

                rows.append({
                    "task": task_name,
                    "model": model_name,
                    "split": split_id,
                    "n_train": int(len(tr)),
                    "n_test": int(len(te)),
                    "pos_train": int(np.sum(ytr == 1)),
                    "neg_train": int(np.sum(ytr == 0)),
                    "pos_test": int(np.sum(yte == 1)),
                    "neg_test": int(np.sum(yte == 0)),
                    "auc": float(auc),
                    "kappa": float(kappa),
                    "mcc": float(mcc),
                    "acc": float(acc),
                    "error": ""
                })

            except Exception as e:
                rows.append({
                    "task": task_name,
                    "model": model_name,
                    "split": split_id,
                    "n_train": int(len(tr)),
                    "n_test": int(len(te)),
                    "pos_train": int(np.sum(ytr == 1)),
                    "neg_train": int(np.sum(ytr == 0)),
                    "pos_test": int(np.sum(yte == 1)),
                    "neg_test": int(np.sum(yte == 0)),
                    "auc": np.nan,
                    "kappa": np.nan,
                    "mcc": np.nan,
                    "acc": np.nan,
                    "error": repr(e)
                })

    res = pd.DataFrame(rows)
    out_all = OUT_DIR / f"{out_prefix}_all_splits.csv"
    out_sum = OUT_DIR / f"{out_prefix}_summary.csv"
    out_best = OUT_DIR / f"{out_prefix}_best_model_refit_full.joblib"

    res.to_csv(out_all, index=False, encoding="utf-8-sig")

    res_ok = res.dropna(subset=["auc", "kappa", "mcc", "acc"]).copy()
    if res_ok.empty:
        raise RuntimeError(f"{task_name}: 所有模型评估都失败了，请检查 {out_all} 的 error 列。")

    summary = (
        res_ok.groupby("model", as_index=False)
        .agg(
            auc_mean=("auc", "mean"),
            auc_median=("auc", "median"),
            auc_std=("auc", "std"),
            auc_max=("auc", "max"),
            kappa_mean=("kappa", "mean"),
            mcc_mean=("mcc", "mean"),
            acc_mean=("acc", "mean"),
            n_splits=("split", "count")
        )
        .sort_values(
            by=["auc_mean", "auc_median", "mcc_mean", "kappa_mean", "acc_mean", "auc_max"],
            ascending=[False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )
    summary.to_csv(out_sum, index=False, encoding="utf-8-sig")

    best_model_name = str(summary.iloc[0]["model"])
    print("\n" + "=" * 60)
    print(f"[BEST] {task_name}")
    print("=" * 60)
    print(f"Best model = {best_model_name}")
    print(summary.head(12).to_string(index=False))

    # refit full
    best_model = get_models()[best_model_name]
    best_pipe = build_model_pipe(best_model_name, best_model, preprocess)
    best_pipe.fit(X, y)
    joblib.dump(best_pipe, out_best)

    return best_model_name, out_labeled, out_all, out_sum, out_best


# =========================
# 6) 主流程
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    feature_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "shape", "size (nm)", "surface treatment", "dispersion medium",
        "pH", "temperature/oC", "Substrate "
    ]

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"以下特征列在文件中不存在：{missing_cols}")

    # 完全对齐你 IC50 的 numeric/categorical 设定
    numeric_cols = [
        "contain O", "contain N", "contain P", "contain S", "contain Si",
        "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main  metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]
    categorical_cols = [
        "shape", "surface treatment", "dispersion medium", "Substrate "
    ]

    pd.DataFrame({
        "feature": feature_cols,
        "type": ["numeric" if c in numeric_cols else "categorical" for c in feature_cols]
    }).to_csv(OUT_DIR / "feature_info.csv", index=False, encoding="utf-8-sig")

    groups_all = df[COL_DOI].copy()
    groups_all = groups_all.fillna("").astype(str).str.strip()
    missing_mask = groups_all.eq("")
    groups_all.loc[missing_mask] = [f"__NO_DOI_ROW_{i}__" for i in df.index[missing_mask]]
    groups_all = groups_all.values

    X_all = prepare_features(df, feature_cols, numeric_cols, categorical_cols)
    preprocess = make_preprocess(numeric_cols, categorical_cols)

    # =====================
    # KmLow
    # =====================
    m_km, y_km, thr_km = make_label_median_log10(df[COL_KM], mode="low_good")
    g_km = groups_all[m_km.values]
    print("\n" + "=" * 60)
    print("KmLow 标签信息")
    print("=" * 60)
    print(f"n={len(y_km)} | median(log10(Km))={thr_km:.6f} | pos_rate={y_km.mean():.3f} | groups={len(np.unique(g_km))}")

    run_one_task(
        task_name="SOD_KmLow",
        df=df,
        y_mask=m_km,
        y=y_km,
        groups=g_km,
        X_all=X_all,
        preprocess=preprocess,
        out_prefix="sod_cls_KmLow"
    )

    # =====================
    # VmaxHigh
    # =====================
    m_v, y_v, thr_v = make_label_median_log10(df[COL_VMAX], mode="high_good")
    g_v = groups_all[m_v.values]
    print("\n" + "=" * 60)
    print("VmaxHigh 标签信息")
    print("=" * 60)
    print(f"n={len(y_v)} | median(log10(Vmax))={thr_v:.6f} | pos_rate={y_v.mean():.3f} | groups={len(np.unique(g_v))}")

    run_one_task(
        task_name="SOD_VmaxHigh",
        df=df,
        y_mask=m_v,
        y=y_v,
        groups=g_v,
        X_all=X_all,
        preprocess=preprocess,
        out_prefix="sod_cls_VmaxHigh"
    )

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


KmLow 标签信息
n=19 | median(log10(Km))=-0.229148 | pos_rate=0.526 | groups=11

[Task] SOD_KmLow
样本数: 19
唯一 DOI 组数: 11
尝试分割次数: 10000
收集到的有效 split 数: 42

[Running] SOD_KmLow | LogReg ...

[Running] SOD_KmLow | LinearSVC ...

[Running] SOD_KmLow | SVC_RBF ...

[Running] SOD_KmLow | KNN ...

[Running] SOD_KmLow | GaussianNB ...

[Running] SOD_KmLow | DecisionTree ...

[Running] SOD_KmLow | RF ...

[Running] SOD_KmLow | ExtraTrees ...

[Running] SOD_KmLow | GradientBoosting ...

[Running] SOD_KmLow | HistGB ...

[Running] SOD_KmLow | AdaBoost ...

[Running] SOD_KmLow | XGBoost ...

[BEST] SOD_KmLow
Best model = KNN
           model  auc_mean  auc_median  auc_std  auc_max  kappa_mean  mcc_mean  acc_mean  n_splits
             KNN  0.833598    1.000000 0.220532     1.00    0.454412  0.492325  0.732644        42
          LogReg  0.714087    0.666667 0.238695     1.00    0.152218  0.159116  0.522855        42
         SVC_RBF  0.659127    0.666667 0.314065     1.00   -0.038742 -0.074376  0.34862

In [17]:
# -*- coding: utf-8 -*-
"""
SOD: KmLow(KNN) & VmaxHigh(XGBoost)
全量数据重训 + SHAP分析 + XGBoost内部重要性(gain/weight)

对齐前面 IC50 的处理细节：
- feature_cols 固定 27 个
- numeric_cols 包含 '3rd metal type'
- categorical_cols 只有 4 个
- clean_text_series 同前
- preprocess: numeric median+indicator + categorical onehot(dense)
- KNN: preprocess -> scaler -> model
- XGB: preprocess -> model

输出目录：
C:\\Users\\86158\\Desktop\\sod_km_vmax_shap_explain

输出内容：
- model_km_knn_refit_full.joblib
- model_vmax_xgb_refit_full.joblib

KmLow:
- km_shap_summary_beeswarm.png
- km_shap_summary_bar.png
- km_shap_mean_abs_per_transformed_feature.csv
- km_shap_mean_abs_agg_by_original_feature.csv

VmaxHigh:
- vmax_xgb_internal_importance.csv
- vmax_xgb_internal_gain.png
- vmax_xgb_internal_weight.png
- vmax_shap_summary_beeswarm.png
- vmax_shap_summary_bar.png
- vmax_shap_mean_abs_per_transformed_feature.csv
- vmax_shap_mean_abs_agg_by_original_feature.csv
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False


# =========================
# 0) 配置
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_km_vmax_shap_explain")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42

# KNN Kernel SHAP 参数（可根据速度调整）
KNN_BACKGROUND_SIZE = 10
KNN_NSAMPLES = 100


# =========================
# 1) 特征定义（对齐前面 IC50）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 2) 清洗 / 预处理
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    # 数值列强制转数值
    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 对齐前面逻辑：3rd metal 三列缺失补 0（含 3rd metal type）
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    # 分类列清洗
    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )
    return preprocess


# =========================
# 3) 标签（中位数切分）
# =========================
def make_label_median_log10(series: pd.Series, mode: str):
    """
    mode:
      - low_good:  log10(x) <= median -> 1
      - high_good: log10(x) >= median -> 1
    """
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))

    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)

    return mask, y, thr


# =========================
# 4) 构建最优模型
# =========================
def build_knn_pipeline(preprocess):
    return Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="distance"))
    ])


def build_xgb_pipeline(preprocess):
    if not HAS_XGBOOST:
        raise RuntimeError("未检测到 xgboost，请先安装：pip install xgboost")

    model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    return Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])


# =========================
# 5) SHAP工具
# =========================
def normalize_shap_output(shap_values):
    """
    兼容不同 shap 版本的二分类输出格式
    返回 shape = (n_samples, n_features) 的正类 shap 值
    """
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return np.asarray(shap_values[1])
        return np.asarray(shap_values[0])

    arr = np.asarray(shap_values)

    # 可能是 (n, p, 2)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]

    return arr


def base_feature_from_transformed(name: str):
    """
    将 preprocess 后的特征名映射回原始特征名，便于聚合
    兼容：
    - num__<col>
    - num__missingindicator_<col>
    - cat__<col>_<level>
    """
    n = str(name)

    if "__" in n:
        n = n.split("__", 1)[1]

    n = n.replace("missingindicator_", "")

    for c in CATEGORICAL_COLS:
        if n == c or n.startswith(c + "_"):
            return c

    for c in NUMERIC_COLS:
        if n == c:
            return c

    return n.split("_")[0]


def save_shap_outputs(shap_values_2d, X_matrix, feature_names, prefix):
    """
    保存：
    - shap beeswarm
    - shap bar
    - 每个 transformed feature 的 mean(|SHAP|)
    - 聚合到原始特征后的 mean(|SHAP|)
    """
    mean_abs = np.mean(np.abs(shap_values_2d), axis=0)

    shap_df = pd.DataFrame({
        "feature_transformed": feature_names,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)

    shap_df.to_csv(
        OUT_DIR / f"{prefix}_shap_mean_abs_per_transformed_feature.csv",
        index=False, encoding="utf-8-sig"
    )

    shap_df["feature_original"] = shap_df["feature_transformed"].apply(base_feature_from_transformed)
    agg = (
        shap_df.groupby("feature_original", as_index=False)["mean_abs_shap"]
        .sum()
        .sort_values("mean_abs_shap", ascending=False)
    )

    agg.to_csv(
        OUT_DIR / f"{prefix}_shap_mean_abs_agg_by_original_feature.csv",
        index=False, encoding="utf-8-sig"
    )

    if HAS_SHAP:
        plt.figure()
        shap.summary_plot(
            shap_values_2d,
            X_matrix,
            feature_names=feature_names,
            show=False,
            max_display=20
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_shap_summary_beeswarm.png", dpi=200)
        plt.close()

        plt.figure()
        shap.summary_plot(
            shap_values_2d,
            X_matrix,
            feature_names=feature_names,
            plot_type="bar",
            show=False,
            max_display=20
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_shap_summary_bar.png", dpi=200)
        plt.close()


# =========================
# 6) KNN 的 SHAP（Kernel SHAP）
# =========================
def run_knn_shap(pipe, X_raw, prefix="km"):
    if not HAS_SHAP:
        print("[Warn] shap 未安装，跳过 KNN SHAP。请先 pip install shap")
        return

    pre = pipe.named_steps["preprocess"]
    scaler = pipe.named_steps["scaler"]
    model = pipe.named_steps["model"]

    X_t = pre.transform(X_raw)
    X_s = scaler.transform(X_t)

    feature_names = [str(x) for x in pre.get_feature_names_out()]

    bg_size = min(KNN_BACKGROUND_SIZE, len(X_s))
    background = shap.sample(X_s, bg_size, random_state=RANDOM_SEED)

    explainer = shap.KernelExplainer(model.predict_proba, background)
    shap_values = explainer.shap_values(X_s, nsamples=KNN_NSAMPLES)
    shap_values_2d = normalize_shap_output(shap_values)

    save_shap_outputs(shap_values_2d, X_s, feature_names, prefix=prefix)


# =========================
# 7) XGBoost 的 SHAP + 内部重要性
# =========================
def run_xgb_internal_and_shap(pipe, X_raw, prefix="vmax"):
    pre = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]

    X_t = pre.transform(X_raw)
    feature_names = [str(x) for x in pre.get_feature_names_out()]

    # ---------- 内部重要性 ----------
    booster = model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")
    weight_dict = booster.get_score(importance_type="weight")

    def fkey_to_name(fk):
        try:
            idx = int(fk[1:])
            return feature_names[idx]
        except Exception:
            return fk

    rows = []
    all_keys = sorted(set(list(gain_dict.keys()) + list(weight_dict.keys())))
    for k in all_keys:
        rows.append({
            "feature_transformed": fkey_to_name(k),
            "gain": float(gain_dict.get(k, 0.0)),
            "weight": float(weight_dict.get(k, 0.0))
        })

    imp_df = pd.DataFrame(rows).sort_values("gain", ascending=False)
    imp_df.to_csv(
        OUT_DIR / f"{prefix}_xgb_internal_importance.csv",
        index=False, encoding="utf-8-sig"
    )

    # gain 图
    top_gain = imp_df.sort_values("gain", ascending=False).head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_gain["feature_transformed"], top_gain["gain"])
    plt.xlabel("XGBoost gain")
    plt.title("Top 20 transformed features (gain)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{prefix}_xgb_internal_gain.png", dpi=200)
    plt.close()

    # weight 图（频率）
    top_weight = imp_df.sort_values("weight", ascending=False).head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_weight["feature_transformed"], top_weight["weight"])
    plt.xlabel("XGBoost weight (frequency)")
    plt.title("Top 20 transformed features (weight/frequency)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{prefix}_xgb_internal_weight.png", dpi=200)
    plt.close()

    # ---------- SHAP ----------
    if not HAS_SHAP:
        print("[Warn] shap 未安装，跳过 XGBoost SHAP。请先 pip install shap")
        return

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_t)
    shap_values_2d = normalize_shap_output(shap_values)

    save_shap_outputs(shap_values_2d, X_t, feature_names, prefix=prefix)


# =========================
# 8) 主流程
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)

    needed = FEATURE_COLS + [COL_KM, COL_VMAX]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"缺少列: {missing}")

    preprocess = make_preprocess()

    # ========= KmLow =========
    m_km, y_km, thr_km = make_label_median_log10(df[COL_KM], mode="low_good")
    df_km = df.loc[m_km].copy()
    X_km = prepare_features(df_km)
    y_km = y_km.astype(int)

    print("=" * 60)
    print("KmLow")
    print("=" * 60)
    print(f"n={len(y_km)} | median(log10(Km))={thr_km:.6f} | pos_rate={y_km.mean():.3f}")

    km_pipe = build_knn_pipeline(preprocess)
    km_pipe.fit(X_km, y_km)
    km_prob = km_pipe.predict_proba(X_km)[:, 1]
    print(f"[KmLow] train AUC (sanity): {roc_auc_score(y_km, km_prob):.4f}")

    joblib.dump(km_pipe, OUT_DIR / "model_km_knn_refit_full.joblib")
    run_knn_shap(km_pipe, X_km, prefix="km")

    # ========= VmaxHigh =========
    m_v, y_v, thr_v = make_label_median_log10(df[COL_VMAX], mode="high_good")
    df_v = df.loc[m_v].copy()
    X_v = prepare_features(df_v)
    y_v = y_v.astype(int)

    print("\n" + "=" * 60)
    print("VmaxHigh")
    print("=" * 60)
    print(f"n={len(y_v)} | median(log10(Vmax))={thr_v:.6f} | pos_rate={y_v.mean():.3f}")

    vmax_pipe = build_xgb_pipeline(preprocess)
    vmax_pipe.fit(X_v, y_v)
    vmax_prob = vmax_pipe.predict_proba(X_v)[:, 1]
    print(f"[VmaxHigh] train AUC (sanity): {roc_auc_score(y_v, vmax_prob):.4f}")

    joblib.dump(vmax_pipe, OUT_DIR / "model_vmax_xgb_refit_full.joblib")
    run_xgb_internal_and_shap(vmax_pipe, X_v, prefix="vmax")

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()

KmLow
n=19 | median(log10(Km))=-0.229148 | pos_rate=0.526
[KmLow] train AUC (sanity): 1.0000


  0%|          | 0/19 [00:00<?, ?it/s]


VmaxHigh
n=18 | median(log10(Vmax))=3.388351 | pos_rate=0.500
[VmaxHigh] train AUC (sanity): 1.0000

Saved to: C:\Users\86158\Desktop\sod_km_vmax_shap_explain


In [22]:
# -*- coding: utf-8 -*-
"""
SOD IC50 regression (uM / ug/mL separated) with DOI-GroupKFold OOF.
Multi-model + hierarchical parameter search (Stage1 coarse -> Stage2 fine) + save best model.

Target:
  y = log10(IC50_value)

Pipeline (configurable):
  preprocess(num+cat+rare+onehot)
    -> VarianceThreshold
    -> (optional) SelectKBest(f_regression)
    -> (optional) TruncatedSVD
    -> (optional) StandardScaler
    -> regressor (Ridge/ElasticNet/BayesianRidge/Huber/SVR/KRR)

Outputs (per unit):
  - <unit>_search_summary.csv               (all tried configs, stage1+stage2)
  - <unit>_best_config.json                (best params + metrics)
  - <unit>_oof_predictions_groupkfold.csv  (best OOF predictions, DOI-GroupKFold)
  - <unit>_best_model_refit_full.joblib    (best pipeline refit on full unit data)

Also:
  - used_data.csv                          (parsed & filtered IC50 rows)
  - feature_info.csv                       (feature type table)

Notes:
  - Column names are normalized: strip + compress multiple spaces.
  - RareCategoryGrouper is fitted inside each fold (no leakage).
  - k_best and svd_dim are made "safe" automatically to avoid invalid settings.
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================
# 0) Paths & global config
# =========================
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"   # <-- 改成你的
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_ic50_reg_doi_multistage_out")  # <-- 改成你的
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IC50 = "IC50/μM"
COL_DOI  = "data reference doi"

RANDOM_SEED = 42
MAX_FOLDS = 5

# -------- Search control --------
# True: 更快（推荐先跑通看结果）；False: 网格更大更慢
FAST_MODE = False

# Stage2：每个 unit 只对 Stage1 里最强的 TopK 模型族做精筛
TOPK_MODEL_FAMILIES_STAGE2 = 1


# =========================
# 1) Column normalize + IC50 parsing
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """strip + 多空格压成单空格（解决 main  metal proportion / Substrate 这种坑）"""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def parse_ic50_cell(x):
    """
    Parse IC50 cell:
      - pure number -> default uM
      - 'value + uM/μM/µM' -> uM
      - 'value + ug/mL/μg/mL/µg/mL' -> ug/mL
    Return: (value, unit)
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================
# 2) Feature preprocessing
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal missing -> 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Fold-wise rare category grouping: freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=3, sparse=True):
    """
    Use sparse OneHot by default.
    Later steps can keep sparse, or densify via SVD / ToDense depending on config.
    """
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


class ToDense(BaseEstimator, TransformerMixin):
    """Convert sparse matrix -> dense ndarray. Safe for small datasets."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    """
    Safe SelectKBest: if requested k > n_features, auto shrink to n_features.
    If k is None, pass-through.
    """
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None
        self.k_effective_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            self.k_effective_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            self.k_effective_ = None
            return self
        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        self.k_effective_ = k_eff
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    """
    Safe TruncatedSVD: if requested n_components >= n_features, auto shrink to n_features-1.
    If n_components is None, pass-through.
    """
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None
        self.n_components_effective_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        n_feat = X.shape[1]
        n_comp = int(self.n_components)
        n_eff = min(n_comp, max(1, n_feat - 1))
        if n_feat <= 1:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        self.n_components_effective_ = n_eff
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 3) Metrics + OOF
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for _, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 4) Model factory + grids (Stage1/Stage2)
# =========================
def instantiate_model(model_name, params):
    if model_name == "Ridge":
        return Ridge(random_state=RANDOM_SEED, **params)
    if model_name == "ElasticNet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_SEED, **params)
    if model_name == "BayesianRidge":
        return BayesianRidge(**params)
    if model_name == "Huber":
        return HuberRegressor(**params)
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def stage1_model_grids(fast=True):
    """Coarse grids for model hyperparams."""
    if fast:
        return {
            "Ridge": [{"alpha": a} for a in [1, 10, 100]],
            "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.01, 0.1] for r in [0.2, 0.5, 0.8]],
            "BayesianRidge": [{}],
            "Huber": [{"alpha": 1e-4, "epsilon": e} for e in [1.35, 1.5]],
            "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                        for C in [10, 30]
                        for g in ["scale", 0.03, 0.1]
                        for e in [0.05, 0.1]],
            "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                                for a in [0.1, 1, 10]
                                for g in [0.03, 0.1]],
        }
    # slower / larger
    return {
        "Ridge": [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]],
        "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]],
        "BayesianRidge": [{}],
        "Huber": [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]],
        "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                    for C in [3, 10, 30, 100]
                    for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                    for e in [0.03, 0.05, 0.1, 0.2]],
        "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                            for a in [0.01, 0.1, 1, 10, 100]
                            for g in [0.01, 0.03, 0.1, 0.3]],
    }


def stage2_model_grids(model_name):
    """Finer grid around the best families."""
    if model_name == "Ridge":
        return [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]]
    if model_name == "ElasticNet":
        return [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]]
    if model_name == "BayesianRidge":
        return [{}]
    if model_name == "Huber":
        return [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]]
    if model_name == "SVR_RBF":
        return [{"C": C, "gamma": g, "epsilon": e}
                for C in [3, 10, 30, 100]
                for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                for e in [0.03, 0.05, 0.1, 0.2]]
    if model_name == "KernelRidge_RBF":
        return [{"alpha": a, "gamma": g}
                for a in [0.01, 0.1, 1, 10, 100]
                for g in [0.01, 0.03, 0.1, 0.3]]
    raise ValueError(model_name)


def stage1_preprocess_grid(fast=True):
    """Coarse grids for preprocessing knobs."""
    if fast:
        return {
            "min_freq_cat": [2, 3],
            "var_thr": [0.0, 1e-5],
            "k_best": [None, 12],
            "svd_dim": [None, 8, 10],
        }
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


def stage2_preprocess_grid():
    """Finer preprocessing grid (still not too huge)."""
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


# =========================
# 5) Build pipeline for one config
# =========================
def build_pipeline_for_config(numeric_cols, categorical_cols,
                             min_freq_cat, var_thr, k_best, svd_dim,
                             model_name, model_params):
    """
    Rules:
      - Sparse OneHot by default.
      - If svd_dim is provided: SVD will densify, so scaler(with_mean=True) ok.
      - If no SVD:
          - keep sparse, use scaler(with_mean=False) for sparse-friendly.
          - for models that need dense, insert ToDense before scaler/model.
    """
    # which models need dense input?
    needs_dense = model_name in {"BayesianRidge", "Huber", "SVR_RBF", "KernelRidge_RBF"}

    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat, sparse=True)

    steps = []
    steps.append(("preprocess", preprocess))
    steps.append(("var", VarianceThreshold(threshold=float(var_thr))))

    # supervised filter
    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(k_best))))

    # dim reduction
    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(svd_dim), random_state=RANDOM_SEED)))
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        # if downstream model needs dense, densify here
        if needs_dense:
            steps.append(("todense", ToDense()))
            steps.append(("scaler", StandardScaler(with_mean=True)))
        else:
            # sparse-friendly scaling
            steps.append(("scaler", StandardScaler(with_mean=False)))

    steps.append(("model", instantiate_model(model_name, model_params)))

    return Pipeline(steps)


# =========================
# 6) Search (Stage1 -> shortlist -> Stage2)
# =========================
def run_unit(unit_name, df_unit, feature_cols, numeric_cols, categorical_cols):
    df_unit = df_unit.copy()

    if len(df_unit) < 10:
        print(f"[Skip] {unit_name}: too few samples: {len(df_unit)}")
        return None

    # y
    y_log = np.log10(df_unit["IC50_value"].astype(float).values)

    # DOI groups
    groups = make_groups(df_unit, COL_DOI)

    # X
    X_raw = prepare_features(df_unit, feature_cols, numeric_cols, categorical_cols)

    # simple stats for you
    n_groups = int(len(np.unique(groups)))
    singleton_groups = int(pd.Series(groups).value_counts().eq(1).sum())
    print(f"\n[{unit_name}] n_samples={len(df_unit)} | n_groups={n_groups} | singleton_groups={singleton_groups}")

    all_rows = []

    # -------- Stage 1 --------
    pre_grid = stage1_preprocess_grid(fast=FAST_MODE)
    model_grids = stage1_model_grids(fast=FAST_MODE)

    for pre_params in ParameterGrid(pre_grid):
        min_freq_cat = pre_params["min_freq_cat"]
        var_thr = pre_params["var_thr"]
        k_best = pre_params["k_best"]
        svd_dim = pre_params["svd_dim"]

        # small logical prune: if both kbest and svd given, svd_dim should be < k_best
        if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
            continue

        for model_name, grid_list in model_grids.items():
            for model_params in grid_list:
                def builder():
                    pipe = build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )
                    return pipe

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 1,
                    "unit": unit_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_unit)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    res_stage1 = pd.DataFrame(all_rows)
    if res_stage1.empty:
        raise RuntimeError(f"{unit_name}: no valid results in stage1 (too few DOI groups?)")

    # shortlist best model families
    fam_best = (
        res_stage1.sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
        .groupby("model", as_index=False)
        .head(1)
        .sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    )
    shortlist = fam_best["model"].tolist()[:TOPK_MODEL_FAMILIES_STAGE2]
    print(f"[{unit_name}] Stage1 shortlist -> {shortlist}")

    # -------- Stage 2 --------
    pre_grid2 = stage2_preprocess_grid()
    for model_name in shortlist:
        for pre_params in ParameterGrid(pre_grid2):
            min_freq_cat = pre_params["min_freq_cat"]
            var_thr = pre_params["var_thr"]
            k_best = pre_params["k_best"]
            svd_dim = pre_params["svd_dim"]
            if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
                continue

            for model_params in stage2_model_grids(model_name):
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 2,
                    "unit": unit_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_unit)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    # -------- Select best overall (prefer stage2 if exists) --------
    res_all = pd.DataFrame(all_rows)
    res_all = res_all.sort_values(
        by=["stage", "r2_log_oof", "rmse_log_oof", "mae_log_oof"],
        ascending=[True, False, True, True]
    ).reset_index(drop=True)

    # save search summary
    res_all.to_csv(OUT_DIR / f"{unit_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    # pick best (stage2 first if available)
    res_stage2 = res_all[res_all["stage"] == 2].copy()
    base = res_stage2 if not res_stage2.empty else res_all
    best = base.sort_values(by=["r2_log_oof", "rmse_log_oof"], ascending=[False, True]).iloc[0].to_dict()

    best_model = best["model"]
    best_min_freq = int(best["min_freq_cat"])
    best_var_thr = float(best["var_thr"])
    best_kbest = None if pd.isna(best["k_best_req"]) else int(best["k_best_req"])
    best_svd = None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"])
    best_model_params = json.loads(best["model_params"])

    # build best builder
    def best_builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            min_freq_cat=best_min_freq,
            var_thr=best_var_thr,
            k_best=best_kbest,
            svd_dim=best_svd,
            model_name=best_model,
            model_params=best_model_params
        )

    # best OOF
    oof, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    oof_df = pd.DataFrame({
        "row_index": df_unit.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof
    })
    oof_df.to_csv(OUT_DIR / f"{unit_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    # refit full + save
    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    best_config = {
        "unit": unit_name,
        "target": "log10(IC50)",
        "split": "GroupKFold by DOI",
        "data_stats": {
            "n_samples": int(len(df_unit)),
            "n_groups": int(n_groups),
            "singleton_groups": int(singleton_groups),
            "n_folds": int(n_folds),
        },
        "best": {
            "stage": int(best["stage"]),
            "model": best_model,
            "preprocess_params": {
                "min_freq_cat": best_min_freq,
                "var_thr": best_var_thr,
                "k_best_req": best_kbest,
                "svd_dim_req": best_svd,
            },
            "model_params": best_model_params,
            "metrics_oof": {
                "r2_log_oof": float(best["r2_log_oof"]),
                "mae_log_oof": float(best["mae_log_oof"]),
                "rmse_log_oof": float(best["rmse_log_oof"]),
                "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
            }
        },
        "features": {
            "feature_cols": feature_cols,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
        }
    }

    with open(OUT_DIR / f"{unit_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "unit": unit_name,
        "pipeline": final_pipe,
        "best_config": best_config
    }, OUT_DIR / f"{unit_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {unit_name} | stage={best_config['best']['stage']} | model={best_model}")
    print("preprocess:", best_config["best"]["preprocess_params"])
    print("model_params:", best_model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)

    return best_config


# =========================
# 7) Main
# =========================
def main():
    df = pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_IC50 not in df.columns:
        raise KeyError(f"IC50 column not found: {COL_IC50}")
    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")

    parsed = df[COL_IC50].apply(parse_ic50_cell)
    df["IC50_value"] = parsed.apply(lambda t: t[0])
    df["IC50_unit"] = parsed.apply(lambda t: t[1])

    df_use = df[
        df["IC50_unit"].isin(["uM", "ug/mL"]) &
        df["IC50_value"].notna() &
        (pd.to_numeric(df["IC50_value"], errors="coerce") > 0)
    ].copy()

    if df_use.empty:
        raise RuntimeError("No valid IC50 rows after parsing.")

    # save used data
    df_use.to_csv(OUT_DIR / "used_data.csv", index=False, encoding="utf-8-sig")

    # -------- canonical feature names (after normalize_columns) --------
    FEATURE_COLS = [
        "contain O","contain N","contain P","contain S","contain Si",
        "contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "shape","size (nm)","surface treatment","dispersion medium",
        "pH","temperature/oC","Substrate"
    ]

    NUMERIC_COLS = [
        "contain O","contain N","contain P","contain S","contain Si",
        "contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "size (nm)","pH","temperature/oC"
    ]

    CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]

    # intersect with existing columns (robust to missing)
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df_use.columns]
    NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df_use.columns]
    CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df_use.columns]

    # save feature info
    feat_info = pd.DataFrame({
        "feature": FEATURE_COLS,
        "type": ["numeric" if c in NUMERIC_COLS else "categorical" for c in FEATURE_COLS]
    })
    feat_info.to_csv(OUT_DIR / "feature_info.csv", index=False, encoding="utf-8-sig")

    # run by unit
    configs = {}
    configs["uM"] = run_unit("uM", df_use[df_use["IC50_unit"] == "uM"].copy(),
                             FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS)

    configs["ugmL"] = run_unit("ugmL", df_use[df_use["IC50_unit"] == "ug/mL"].copy(),
                               FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS)

    print("\nSaved to:", OUT_DIR)
    return configs


if __name__ == "__main__":
    main()


[uM] n_samples=27 | n_groups=12 | singleton_groups=6
[uM] Stage1 shortlist -> ['SVR_RBF']

[BEST] uM | stage=2 | model=SVR_RBF
preprocess: {'min_freq_cat': 3, 'var_thr': 0.0, 'k_best_req': None, 'svd_dim_req': 12}
model_params: {'C': 100, 'gamma': 'scale', 'epsilon': 0.2}
metrics_oof: {'r2_log_oof': 0.05878085037190994, 'mae_log_oof': 1.596645209644892, 'rmse_log_oof': 2.7512555316620535, 'pearson_r_oof': 0.26532888196748605, 'pearson_r2_oof': 0.07039941560611615}

[ugmL] n_samples=23 | n_groups=13 | singleton_groups=9
[ugmL] Stage1 shortlist -> ['SVR_RBF']

[BEST] ugmL | stage=2 | model=SVR_RBF
preprocess: {'min_freq_cat': 3, 'var_thr': 0.0, 'k_best_req': None, 'svd_dim_req': 10}
model_params: {'C': 30, 'gamma': 0.01, 'epsilon': 0.2}
metrics_oof: {'r2_log_oof': 0.600256961329644, 'mae_log_oof': 0.598497399689025, 'rmse_log_oof': 0.759539894863608, 'pearson_r_oof': 0.8017164978828983, 'pearson_r2_oof': 0.6427493429776193}

Saved to: C:\Users\86158\Desktop\sod_ic50_reg_doi_multistage_o

In [1]:
# -*- coding: utf-8 -*-
"""
SOD regression: logKm / logVmax (separately) with DOI-GroupKFold OOF.
Multi-model + hierarchical parameter search (Stage1 coarse -> Stage2 fine) + save best model.

Targets:
  - logKm   = log10(Km/mM)
  - logVmax = log10(Vmax/μM s-1)

Pipeline (configurable):
  preprocess(num+cat+rare+onehot)
    -> VarianceThreshold
    -> (optional) SelectKBest(f_regression)
    -> (optional) TruncatedSVD
    -> (optional) StandardScaler
    -> regressor (Ridge/ElasticNet/BayesianRidge/Huber/SVR/KRR)

Outputs (per target):
  - <target>_search_summary.csv
  - <target>_best_config.json
  - <target>_oof_predictions_groupkfold.csv
  - <target>_best_model_refit_full.joblib

Also:
  - used_data_logKm.csv / used_data_logVmax.csv
  - feature_info.csv

Notes:
  - Column names normalized: strip + compress multiple spaces.
  - RareCategoryGrouper fitted fold-wise (no leakage).
  - k_best / svd_dim safe-adjusted to avoid invalid settings.
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================
# 0) Paths & global config
# =========================
SOD_PATH = Path(r"C:\Users\86158\Desktop\dataissod.xlsx")   # <-- 改成你的输入文件
SHEET_NAME = 0

# 输出目录自动放在输入 Excel 同级目录下
OUT_DIR = SOD_PATH.parent / "sod_km_vmax_reg_doi_multistage_out"

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5

# True: 更快（推荐先跑通）；False: 网格更大更慢
FAST_MODE = False

# Stage2：每个 target 只对 Stage1 里最强的 TopK 模型族做精筛
TOPK_MODEL_FAMILIES_STAGE2 = 1


# =========================
# 0.1) Path / write checks
# =========================
def ensure_input_file_exists(file_path: Path):
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")
    if not file_path.is_file():
        raise FileNotFoundError(f"Input path is not a file: {file_path}")


def check_output_dir_writable(out_dir: Path):
    """
    先检查目录能否创建，且是否能正确写入 txt / csv / json / joblib
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    test_txt = out_dir / "_write_test.txt"
    test_csv = out_dir / "_write_test.csv"
    test_json = out_dir / "_write_test.json"
    test_joblib = out_dir / "_write_test.joblib"

    try:
        with open(test_txt, "w", encoding="utf-8") as f:
            f.write("write ok")

        pd.DataFrame({"status": ["ok"], "value": [1]}).to_csv(
            test_csv, index=False, encoding="utf-8-sig"
        )

        with open(test_json, "w", encoding="utf-8") as f:
            json.dump({"status": "ok", "value": 1}, f, ensure_ascii=False, indent=2)

        joblib.dump({"status": "ok", "value": 1}, test_joblib)

        print(f"[OK] Output directory writable: {out_dir}")
        print("[OK] Write test passed for txt/csv/json/joblib")

    except Exception as e:
        raise OSError(f"Output directory is not writable: {out_dir}\nReason: {e}")

    finally:
        for p in [test_txt, test_csv, test_json, test_joblib]:
            try:
                if p.exists():
                    p.unlink()
            except Exception:
                pass


# =========================
# 1) Column normalize
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """strip + 多空格压成单空格（解决 main  metal proportion / Substrate 这种坑）"""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


# =========================
# 2) Feature preprocessing
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal missing -> 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Fold-wise rare category grouping: freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


class ToDense(BaseEstimator, TransformerMixin):
    """Convert sparse matrix -> dense ndarray (ok for small datasets)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    """Safe SelectKBest: auto shrink k if k > n_features. If k is None -> pass-through."""
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None
        self.k_effective_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            self.k_effective_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            self.k_effective_ = None
            return self
        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        self.k_effective_ = k_eff
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    """Safe TruncatedSVD: auto shrink n_components to n_features-1. If None -> pass-through."""
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None
        self.n_components_effective_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        n_feat = X.shape[1]
        n_comp = int(self.n_components)
        n_eff = min(n_comp, max(1, n_feat - 1))
        if n_feat <= 1:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        self.n_components_effective_ = n_eff
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 3) Metrics + OOF
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for _, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 4) Model factory + grids (Stage1/Stage2)
# =========================
def instantiate_model(model_name, params):
    if model_name == "Ridge":
        return Ridge(random_state=RANDOM_SEED, **params)
    if model_name == "ElasticNet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_SEED, **params)
    if model_name == "BayesianRidge":
        return BayesianRidge(**params)
    if model_name == "Huber":
        return HuberRegressor(**params)
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def stage1_model_grids(fast=True):
    if fast:
        return {
            "Ridge": [{"alpha": a} for a in [1, 10, 100]],
            "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.01, 0.1] for r in [0.2, 0.5, 0.8]],
            "BayesianRidge": [{}],
            "Huber": [{"alpha": 1e-4, "epsilon": e} for e in [1.35, 1.5]],
            "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                        for C in [10, 30]
                        for g in ["scale", 0.03, 0.1]
                        for e in [0.05, 0.1]],
            "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                                for a in [0.1, 1, 10]
                                for g in [0.03, 0.1]],
        }
    return {
        "Ridge": [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]],
        "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]],
        "BayesianRidge": [{}],
        "Huber": [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]],
        "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                    for C in [3, 10, 30, 100]
                    for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                    for e in [0.03, 0.05, 0.1, 0.2]],
        "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                            for a in [0.01, 0.1, 1, 10, 100]
                            for g in [0.01, 0.03, 0.1, 0.3]],
    }


def stage2_model_grids(model_name):
    if model_name == "Ridge":
        return [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]]
    if model_name == "ElasticNet":
        return [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]]
    if model_name == "BayesianRidge":
        return [{}]
    if model_name == "Huber":
        return [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]]
    if model_name == "SVR_RBF":
        return [{"C": C, "gamma": g, "epsilon": e}
                for C in [3, 10, 30, 100]
                for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                for e in [0.03, 0.05, 0.1, 0.2]]
    if model_name == "KernelRidge_RBF":
        return [{"alpha": a, "gamma": g}
                for a in [0.01, 0.1, 1, 10, 100]
                for g in [0.01, 0.03, 0.1, 0.3]]
    raise ValueError(model_name)


def stage1_preprocess_grid(fast=True):
    if fast:
        return {
            "min_freq_cat": [2, 3],
            "var_thr": [0.0, 1e-5],
            "k_best": [None, 12],
            "svd_dim": [None, 8, 10],
        }
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


def stage2_preprocess_grid():
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


# =========================
# 5) Build pipeline for one config
# =========================
def build_pipeline_for_config(numeric_cols, categorical_cols,
                             min_freq_cat, var_thr, k_best, svd_dim,
                             model_name, model_params):
    needs_dense = model_name in {"BayesianRidge", "Huber", "SVR_RBF", "KernelRidge_RBF"}

    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat, sparse=True)

    steps = []
    steps.append(("preprocess", preprocess))
    steps.append(("var", VarianceThreshold(threshold=float(var_thr))))

    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(k_best))))

    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(svd_dim), random_state=RANDOM_SEED)))
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        if needs_dense:
            steps.append(("todense", ToDense()))
            steps.append(("scaler", StandardScaler(with_mean=True)))
        else:
            steps.append(("scaler", StandardScaler(with_mean=False)))

    steps.append(("model", instantiate_model(model_name, model_params)))
    return Pipeline(steps)


# =========================
# 6) Search for one target
# =========================
def run_target(target_name, df_target, feature_cols, numeric_cols, categorical_cols, y_col):
    df_target = df_target.copy()

    y_num = pd.to_numeric(df_target[y_col], errors="coerce")
    df_target = df_target[y_num.notna() & (y_num > 0)].copy()
    if len(df_target) < 10:
        print(f"[Skip] {target_name}: too few samples: {len(df_target)}")
        return None

    y_log = np.log10(pd.to_numeric(df_target[y_col], errors="coerce").astype(float).values)

    groups = make_groups(df_target, COL_DOI)
    X_raw = prepare_features(df_target, feature_cols, numeric_cols, categorical_cols)

    n_groups = int(len(np.unique(groups)))
    singleton_groups = int(pd.Series(groups).value_counts().eq(1).sum())
    print(f"\n[{target_name}] n_samples={len(df_target)} | n_groups={n_groups} | singleton_groups={singleton_groups}")

    all_rows = []

    pre_grid = stage1_preprocess_grid(fast=FAST_MODE)
    model_grids = stage1_model_grids(fast=FAST_MODE)

    # -------- Stage 1 --------
    for pre_params in ParameterGrid(pre_grid):
        min_freq_cat = pre_params["min_freq_cat"]
        var_thr = pre_params["var_thr"]
        k_best = pre_params["k_best"]
        svd_dim = pre_params["svd_dim"]

        if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
            continue

        for model_name, grid_list in model_grids.items():
            for model_params in grid_list:
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 1,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    res_stage1 = pd.DataFrame(all_rows)
    if res_stage1.empty:
        raise RuntimeError(f"{target_name}: no valid results in stage1 (too few DOI groups?)")

    fam_best = (
        res_stage1.sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
        .groupby("model", as_index=False)
        .head(1)
        .sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    )
    shortlist = fam_best["model"].tolist()[:TOPK_MODEL_FAMILIES_STAGE2]
    print(f"[{target_name}] Stage1 shortlist -> {shortlist}")

    # -------- Stage 2 --------
    pre_grid2 = stage2_preprocess_grid()
    for model_name in shortlist:
        for pre_params in ParameterGrid(pre_grid2):
            min_freq_cat = pre_params["min_freq_cat"]
            var_thr = pre_params["var_thr"]
            k_best = pre_params["k_best"]
            svd_dim = pre_params["svd_dim"]
            if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
                continue

            for model_params in stage2_model_grids(model_name):
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 2,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    # -------- Select best (prefer stage2) --------
    res_all = pd.DataFrame(all_rows)
    res_all.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    res_stage2 = res_all[res_all["stage"] == 2].copy()
    base = res_stage2 if not res_stage2.empty else res_all
    best = base.sort_values(by=["r2_log_oof", "rmse_log_oof"], ascending=[False, True]).iloc[0].to_dict()

    best_model = best["model"]
    best_min_freq = int(best["min_freq_cat"])
    best_var_thr = float(best["var_thr"])
    best_kbest = None if pd.isna(best["k_best_req"]) else int(best["k_best_req"])
    best_svd = None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"])
    best_model_params = json.loads(best["model_params"])

    def best_builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            min_freq_cat=best_min_freq,
            var_thr=best_var_thr,
            k_best=best_kbest,
            svd_dim=best_svd,
            model_name=best_model,
            model_params=best_model_params
        )

    out_best = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    if out_best is None:
        raise RuntimeError(f"{target_name}: cannot compute OOF for best config.")

    oof, ok, n_folds = out_best

    pd.DataFrame({
        "row_index": df_target.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof
    }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "data_stats": {
            "n_samples": int(len(df_target)),
            "n_groups": int(n_groups),
            "singleton_groups": int(singleton_groups),
            "n_folds": int(n_folds),
        },
        "best": {
            "stage": int(best["stage"]),
            "model": best_model,
            "preprocess_params": {
                "min_freq_cat": best_min_freq,
                "var_thr": best_var_thr,
                "k_best_req": best_kbest,
                "svd_dim_req": best_svd,
            },
            "model_params": best_model_params,
            "metrics_oof": {
                "r2_log_oof": float(best["r2_log_oof"]),
                "mae_log_oof": float(best["mae_log_oof"]),
                "rmse_log_oof": float(best["rmse_log_oof"]),
                "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
            }
        },
        "features": {
            "feature_cols": feature_cols,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
        }
    }

    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "target": target_name,
        "pipeline": final_pipe,
        "best_config": best_config
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | stage={best_config['best']['stage']} | model={best_model}")
    print("preprocess:", best_config["best"]["preprocess_params"])
    print("model_params:", best_model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)

    return best_config


# =========================
# 7) Main
# =========================
def main():
    print("=" * 72)
    print("Input file :", SOD_PATH)
    print("Output dir :", OUT_DIR)
    print("=" * 72)

    # 先检查输入文件和输出目录可写
    ensure_input_file_exists(SOD_PATH)
    check_output_dir_writable(OUT_DIR)

    df = pd.read_excel(str(SOD_PATH), sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")
    if COL_KM not in df.columns:
        raise KeyError(f"Km column not found: {COL_KM}")
    if COL_VMAX not in df.columns:
        raise KeyError(f"Vmax column not found: {COL_VMAX}")

    FEATURE_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "shape","size (nm)","surface treatment","dispersion medium","pH","temperature/oC","Substrate"
    ]
    NUMERIC_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "size (nm)","pH","temperature/oC"
    ]
    CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]

    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]
    CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df.columns]

    pd.DataFrame({
        "feature": FEATURE_COLS,
        "type": ["numeric" if c in NUMERIC_COLS else "categorical" for c in FEATURE_COLS]
    }).to_csv(OUT_DIR / "feature_info.csv", index=False, encoding="utf-8-sig")

    # ---- logKm ----
    df_km = df.copy()
    km_num = pd.to_numeric(df_km[COL_KM], errors="coerce")
    df_km = df_km[km_num.notna() & (km_num > 0)].copy()
    df_km.to_csv(OUT_DIR / "used_data_logKm.csv", index=False, encoding="utf-8-sig")
    run_target("logKm", df_km, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, y_col=COL_KM)

    # ---- logVmax ----
    df_v = df.copy()
    v_num = pd.to_numeric(df_v[COL_VMAX], errors="coerce")
    df_v = df_v[v_num.notna() & (v_num > 0)].copy()
    df_v.to_csv(OUT_DIR / "used_data_logVmax.csv", index=False, encoding="utf-8-sig")
    run_target("logVmax", df_v, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, y_col=COL_VMAX)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()

Input file : C:\Users\86158\Desktop\dataissod.xlsx
Output dir : C:\Users\86158\Desktop\sod_km_vmax_reg_doi_multistage_out
[OK] Output directory writable: C:\Users\86158\Desktop\sod_km_vmax_reg_doi_multistage_out
[OK] Write test passed for txt/csv/json/joblib

[logKm] n_samples=19 | n_groups=11 | singleton_groups=7
[logKm] Stage1 shortlist -> ['SVR_RBF']

[BEST] logKm | stage=2 | model=SVR_RBF
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': None, 'svd_dim_req': 8}
model_params: {'C': 100, 'gamma': 0.03, 'epsilon': 0.03}
metrics_oof: {'r2_log_oof': 0.6791101327697001, 'mae_log_oof': 0.7491162029147429, 'rmse_log_oof': 0.9229648999587763, 'pearson_r_oof': 0.8340824749195995, 'pearson_r2_oof': 0.6956935749680043}

[logVmax] n_samples=18 | n_groups=10 | singleton_groups=6
[logVmax] Stage1 shortlist -> ['SVR_RBF']

[BEST] logVmax | stage=2 | model=SVR_RBF
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': None, 'svd_dim_req': 6}
model_params: {'C': 30, 'gamma': 0.01,

In [2]:
# -*- coding: utf-8 -*-
"""
Plot evaluation figures for SOD logKm regression using saved OOF predictions.

Reads:
- OUT_DIR/logKm_oof_predictions_groupkfold.csv

Writes:
- OUT_DIR/plots_logKm/*.png
"""

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ====== EDIT HERE ======
OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_km_vmax_reg_doi_multistage_out")
TARGET = "logKm"
# =======================


def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def compute_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    r = pearsonr_safe(y_true, y_pred)
    return {"r2": r2, "rmse": rmse, "mae": mae, "pearson_r": r, "pearson_r2": (r*r if not np.isnan(r) else np.nan)}


def set_clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


def save_fig(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=260)
    plt.close(fig)


def qq_plot(ax, residuals):
    """
    Simple QQ plot without scipy:
    - theoretical quantiles from N(0,1) approximated by sorting residuals and using normal CDF inverse via numpy?
    No scipy here, so we use a robust approximation:
    - Use rank-based probabilities and approximate inverse-erf.
    """
    res = np.asarray(residuals, float)
    res = res[np.isfinite(res)]
    res = np.sort(res)
    n = len(res)
    if n < 5:
        ax.text(0.5, 0.5, "Too few points for QQ", ha="center", va="center", transform=ax.transAxes)
        return

    # probabilities
    p = (np.arange(1, n + 1) - 0.5) / n

    # inverse CDF for standard normal using erfinv approximation
    # z = sqrt(2) * erfinv(2p-1)
    # numpy has np.erf but not erfinv in older versions; use np.special if available
    try:
        from numpy import sqrt
        from numpy.special import erfinv
        z = np.sqrt(2.0) * erfinv(2.0 * p - 1.0)
    except Exception:
        # fallback: linear proxy (not perfect but still diagnostic)
        z = np.linspace(-2.5, 2.5, n)

    ax.scatter(z, res, s=22, alpha=0.75)

    # reference line using quartiles
    q1r, q3r = np.quantile(res, [0.25, 0.75])
    q1z, q3z = np.quantile(z, [0.25, 0.75])
    slope = (q3r - q1r) / (q3z - q1z + 1e-12)
    intercept = q1r - slope * q1z
    xx = np.array([z.min(), z.max()])
    ax.plot(xx, intercept + slope * xx, linestyle="--", linewidth=1)

    ax.set_xlabel("Theoretical quantiles (N(0,1))")
    ax.set_ylabel("Residual quantiles")
    ax.set_title("Residual Q-Q plot")


def main():
    oof_path = OUT_DIR / f"{TARGET}_oof_predictions_groupkfold.csv"
    if not oof_path.exists():
        raise FileNotFoundError(f"OOF file not found: {oof_path}")

    df = pd.read_csv(oof_path, encoding="utf-8-sig")
    for col in ["y_log_true", "y_log_oof_pred"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["y_log_true", "y_log_oof_pred"]).reset_index(drop=True)

    y_true = df["y_log_true"].values
    y_pred = df["y_log_oof_pred"].values
    resid = y_pred - y_true

    m = compute_metrics(y_true, y_pred)

    plot_dir = OUT_DIR / "plots_logKm"
    plot_dir.mkdir(parents=True, exist_ok=True)

    # 1) Parity (log10)
    fig = plt.figure(figsize=(6.6, 5.4))
    ax = fig.add_subplot(111)
    ax.scatter(y_true, y_pred, s=28, alpha=0.75)

    mn = float(min(y_true.min(), y_pred.min()))
    mx = float(max(y_true.max(), y_pred.max()))
    pad = 0.06 * (mx - mn + 1e-9)
    lo, hi = mn - pad, mx + pad
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

    ax.set_xlabel("True log10(Km/mM)")
    ax.set_ylabel("OOF Pred log10(Km/mM)")
    ax.set_title("logKm Parity (DOI-GroupKFold OOF)")

    ax.text(
        0.02, 0.98,
        f"R2={m['r2']:.4f}\nRMSE={m['rmse']:.4f}\nMAE={m['mae']:.4f}\nPearson r={m['pearson_r']:.4f}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "parity_log10.png")

    # 2) Residual vs True
    fig = plt.figure(figsize=(6.6, 5.4))
    ax = fig.add_subplot(111)
    ax.scatter(y_true, resid, s=28, alpha=0.75)
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("True log10(Km/mM)")
    ax.set_ylabel("Residual = Pred - True (log10)")
    ax.set_title("logKm Residual vs True (OOF)")
    ax.text(
        0.02, 0.98,
        f"mean(resid)={np.mean(resid):.4f}\nstd(resid)={np.std(resid, ddof=1):.4f}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "residual_vs_true.png")

    # 3) Residual histogram
    fig = plt.figure(figsize=(6.6, 5.4))
    ax = fig.add_subplot(111)
    ax.hist(resid, bins=10)
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Residual = Pred - True (log10)")
    ax.set_ylabel("Count")
    ax.set_title("logKm Residual Histogram (OOF)")
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "residual_hist.png")

    # 4) QQ plot of residuals
    fig = plt.figure(figsize=(6.6, 5.4))
    ax = fig.add_subplot(111)
    qq_plot(ax, resid)
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "qq_residual.png")

    print("[Saved plots]", plot_dir)


if __name__ == "__main__":
    main()

[Saved plots] C:\Users\86158\Desktop\sod_km_vmax_reg_doi_multistage_out\plots_logKm


In [1]:
# -*- coding: utf-8 -*-
"""
SOD/CAT regression: logKcat (separately) with DOI-GroupKFold OOF.
Multi-model + hierarchical parameter search (Stage1 coarse -> Stage2 fine) + save best model.

Targets:
  - SOD: log10(Kcat/s-1)
  - CAT: log10(Kcat/s-1)

Pipeline (configurable):
  preprocess(num+cat+rare+onehot)
    -> VarianceThreshold
    -> (optional) SelectKBest(f_regression)
    -> (optional) TruncatedSVD
    -> (optional) StandardScaler
    -> regressor (Ridge/ElasticNet/BayesianRidge/Huber/SVR/KRR)

Outputs (per dataset):
  - logKcat_search_summary.csv
  - logKcat_best_config.json
  - logKcat_oof_predictions_groupkfold.csv
  - logKcat_best_model_refit_full.joblib
Also:
  - used_data_logKcat.csv
  - feature_info.csv

Notes:
  - Column names normalized: strip + compress multiple spaces.
  - RareCategoryGrouper fitted fold-wise (no leakage).
  - k_best / svd_dim safe-adjusted to avoid invalid settings.
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================
# 0) Paths & global config
# =========================
SOD_PATH = Path(r"C:\Users\86158\Desktop\dataissod.xlsx")   # <-- SOD 文件
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")   # <-- CAT 文件
SHEET_NAME = 0

COL_DOI   = "data reference doi"
COL_KCAT  = "Kcat/s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5

# True: 更快（推荐先跑通）；False: 网格更大更慢
FAST_MODE = False

# Stage2：每个 target 只对 Stage1 里最强的 TopK 模型族做精筛
TOPK_MODEL_FAMILIES_STAGE2 = 1


# =========================
# 0.1) Path / write checks
# =========================
def ensure_input_file_exists(file_path: Path):
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")
    if not file_path.is_file():
        raise FileNotFoundError(f"Input path is not a file: {file_path}")


def check_output_dir_writable(out_dir: Path):
    """
    先检查目录能否创建，且是否能正确写入 txt / csv / json / joblib
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    test_txt = out_dir / "_write_test.txt"
    test_csv = out_dir / "_write_test.csv"
    test_json = out_dir / "_write_test.json"
    test_joblib = out_dir / "_write_test.joblib"

    try:
        with open(test_txt, "w", encoding="utf-8") as f:
            f.write("write ok")

        pd.DataFrame({"status": ["ok"], "value": [1]}).to_csv(
            test_csv, index=False, encoding="utf-8-sig"
        )

        with open(test_json, "w", encoding="utf-8") as f:
            json.dump({"status": "ok", "value": 1}, f, ensure_ascii=False, indent=2)

        joblib.dump({"status": "ok", "value": 1}, test_joblib)

        print(f"[OK] Output directory writable: {out_dir}")
        print("[OK] Write test passed for txt/csv/json/joblib")

    except Exception as e:
        raise OSError(f"Output directory is not writable: {out_dir}\nReason: {e}")

    finally:
        for p in [test_txt, test_csv, test_json, test_joblib]:
            try:
                if p.exists():
                    p.unlink()
            except Exception:
                pass


# =========================
# 1) Column normalize
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """strip + 多空格压成单空格（解决 main  metal proportion / Substrate 这种坑）"""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


# =========================
# 2) Feature preprocessing
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal missing -> 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Fold-wise rare category grouping: freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


class ToDense(BaseEstimator, TransformerMixin):
    """Convert sparse matrix -> dense ndarray (ok for small datasets)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    """Safe SelectKBest: auto shrink k if k > n_features. If k is None -> pass-through."""
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None
        self.k_effective_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            self.k_effective_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            self.k_effective_ = None
            return self
        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        self.k_effective_ = k_eff
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    """Safe TruncatedSVD: auto shrink n_components to n_features-1. If None -> pass-through."""
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None
        self.n_components_effective_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        n_feat = X.shape[1]
        n_comp = int(self.n_components)
        n_eff = min(n_comp, max(1, n_feat - 1))
        if n_feat <= 1:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        self.n_components_effective_ = n_eff
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 3) Metrics + OOF
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for _, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 4) Model factory + grids (Stage1/Stage2)
# =========================
def instantiate_model(model_name, params):
    if model_name == "Ridge":
        return Ridge(random_state=RANDOM_SEED, **params)
    if model_name == "ElasticNet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_SEED, **params)
    if model_name == "BayesianRidge":
        return BayesianRidge(**params)
    if model_name == "Huber":
        return HuberRegressor(**params)
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def stage1_model_grids(fast=True):
    if fast:
        return {
            "Ridge": [{"alpha": a} for a in [1, 10, 100]],
            "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.01, 0.1] for r in [0.2, 0.5, 0.8]],
            "BayesianRidge": [{}],
            "Huber": [{"alpha": 1e-4, "epsilon": e} for e in [1.35, 1.5]],
            "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                        for C in [10, 30]
                        for g in ["scale", 0.03, 0.1]
                        for e in [0.05, 0.1]],
            "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                                for a in [0.1, 1, 10]
                                for g in [0.03, 0.1]],
        }
    return {
        "Ridge": [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]],
        "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]],
        "BayesianRidge": [{}],
        "Huber": [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]],
        "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                    for C in [3, 10, 30, 100]
                    for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                    for e in [0.03, 0.05, 0.1, 0.2]],
        "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                            for a in [0.01, 0.1, 1, 10, 100]
                            for g in [0.01, 0.03, 0.1, 0.3]],
    }


def stage2_model_grids(model_name):
    if model_name == "Ridge":
        return [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]]
    if model_name == "ElasticNet":
        return [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]]
    if model_name == "BayesianRidge":
        return [{}]
    if model_name == "Huber":
        return [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]]
    if model_name == "SVR_RBF":
        return [{"C": C, "gamma": g, "epsilon": e}
                for C in [3, 10, 30, 100]
                for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                for e in [0.03, 0.05, 0.1, 0.2]]
    if model_name == "KernelRidge_RBF":
        return [{"alpha": a, "gamma": g}
                for a in [0.01, 0.1, 1, 10, 100]
                for g in [0.01, 0.03, 0.1, 0.3]]
    raise ValueError(model_name)


def stage1_preprocess_grid(fast=True):
    if fast:
        return {
            "min_freq_cat": [2, 3],
            "var_thr": [0.0, 1e-5],
            "k_best": [None, 12],
            "svd_dim": [None, 8, 10],
        }
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


def stage2_preprocess_grid():
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


# =========================
# 5) Build pipeline for one config
# =========================
def build_pipeline_for_config(numeric_cols, categorical_cols,
                             min_freq_cat, var_thr, k_best, svd_dim,
                             model_name, model_params):
    needs_dense = model_name in {"BayesianRidge", "Huber", "SVR_RBF", "KernelRidge_RBF"}

    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat, sparse=True)

    steps = []
    steps.append(("preprocess", preprocess))
    steps.append(("var", VarianceThreshold(threshold=float(var_thr))))

    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(k_best))))

    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(svd_dim), random_state=RANDOM_SEED)))
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        if needs_dense:
            steps.append(("todense", ToDense()))
            steps.append(("scaler", StandardScaler(with_mean=True)))
        else:
            steps.append(("scaler", StandardScaler(with_mean=False)))

    steps.append(("model", instantiate_model(model_name, model_params)))
    return Pipeline(steps)


# =========================
# 6) Search for one target
# =========================
def run_target(target_name, df_target, feature_cols, numeric_cols, categorical_cols, y_col, out_dir):
    df_target = df_target.copy()

    y_num = pd.to_numeric(df_target[y_col], errors="coerce")
    df_target = df_target[y_num.notna() & (y_num > 0)].copy()
    if len(df_target) < 10:
        print(f"[Skip] {target_name}: too few samples: {len(df_target)}")
        return None

    y_log = np.log10(pd.to_numeric(df_target[y_col], errors="coerce").astype(float).values)

    groups = make_groups(df_target, COL_DOI)
    X_raw = prepare_features(df_target, feature_cols, numeric_cols, categorical_cols)

    n_groups = int(len(np.unique(groups)))
    singleton_groups = int(pd.Series(groups).value_counts().eq(1).sum())
    print(f"\n[{target_name}] n_samples={len(df_target)} | n_groups={n_groups} | singleton_groups={singleton_groups}")

    all_rows = []

    pre_grid = stage1_preprocess_grid(fast=FAST_MODE)
    model_grids = stage1_model_grids(fast=FAST_MODE)

    # -------- Stage 1 --------
    for pre_params in ParameterGrid(pre_grid):
        min_freq_cat = pre_params["min_freq_cat"]
        var_thr = pre_params["var_thr"]
        k_best = pre_params["k_best"]
        svd_dim = pre_params["svd_dim"]

        if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
            continue

        for model_name, grid_list in model_grids.items():
            for model_params in grid_list:
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 1,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    res_stage1 = pd.DataFrame(all_rows)
    if res_stage1.empty:
        raise RuntimeError(f"{target_name}: no valid results in stage1 (too few DOI groups?)")

    fam_best = (
        res_stage1.sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
        .groupby("model", as_index=False)
        .head(1)
        .sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    )
    shortlist = fam_best["model"].tolist()[:TOPK_MODEL_FAMILIES_STAGE2]
    print(f"[{target_name}] Stage1 shortlist -> {shortlist}")

    # -------- Stage 2 --------
    pre_grid2 = stage2_preprocess_grid()
    for model_name in shortlist:
        for pre_params in ParameterGrid(pre_grid2):
            min_freq_cat = pre_params["min_freq_cat"]
            var_thr = pre_params["var_thr"]
            k_best = pre_params["k_best"]
            svd_dim = pre_params["svd_dim"]
            if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
                continue

            for model_params in stage2_model_grids(model_name):
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 2,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    # -------- Select best (prefer stage2) --------
    res_all = pd.DataFrame(all_rows)
    res_all.to_csv(out_dir / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    res_stage2 = res_all[res_all["stage"] == 2].copy()
    base = res_stage2 if not res_stage2.empty else res_all
    best = base.sort_values(by=["r2_log_oof", "rmse_log_oof"], ascending=[False, True]).iloc[0].to_dict()

    best_model = best["model"]
    best_min_freq = int(best["min_freq_cat"])
    best_var_thr = float(best["var_thr"])
    best_kbest = None if pd.isna(best["k_best_req"]) else int(best["k_best_req"])
    best_svd = None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"])
    best_model_params = json.loads(best["model_params"])

    def best_builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            min_freq_cat=best_min_freq,
            var_thr=best_var_thr,
            k_best=best_kbest,
            svd_dim=best_svd,
            model_name=best_model,
            model_params=best_model_params
        )

    out_best = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    if out_best is None:
        raise RuntimeError(f"{target_name}: cannot compute OOF for best config.")

    oof, ok, n_folds = out_best

    pd.DataFrame({
        "row_index": df_target.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof
    }).to_csv(out_dir / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "data_stats": {
            "n_samples": int(len(df_target)),
            "n_groups": int(n_groups),
            "singleton_groups": int(singleton_groups),
            "n_folds": int(n_folds),
        },
        "best": {
            "stage": int(best["stage"]),
            "model": best_model,
            "preprocess_params": {
                "min_freq_cat": best_min_freq,
                "var_thr": best_var_thr,
                "k_best_req": best_kbest,
                "svd_dim_req": best_svd,
            },
            "model_params": best_model_params,
            "metrics_oof": {
                "r2_log_oof": float(best["r2_log_oof"]),
                "mae_log_oof": float(best["mae_log_oof"]),
                "rmse_log_oof": float(best["rmse_log_oof"]),
                "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
            }
        },
        "features": {
            "feature_cols": feature_cols,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
        }
    }

    with open(out_dir / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "target": target_name,
        "pipeline": final_pipe,
        "best_config": best_config
    }, out_dir / f"{target_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | stage={best_config['best']['stage']} | model={best_model}")
    print("preprocess:", best_config["best"]["preprocess_params"])
    print("model_params:", best_model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)

    return best_config


# =========================
# 7) Run one dataset
# =========================
def run_one_dataset(dataset_name, input_path: Path):
    out_dir = input_path.parent / f"{dataset_name.lower()}_kcat_reg_doi_multistage_out"

    print("\n" + "#" * 72)
    print(f"Running dataset: {dataset_name}")
    print("Input file :", input_path)
    print("Output dir :", out_dir)
    print("#" * 72)

    ensure_input_file_exists(input_path)
    check_output_dir_writable(out_dir)

    df = pd.read_excel(str(input_path), sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")
    if COL_KCAT not in df.columns:
        raise KeyError(f"Kcat column not found: {COL_KCAT}")

    FEATURE_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "shape","size (nm)","surface treatment","dispersion medium","pH","temperature/oC","Substrate"
    ]
    NUMERIC_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "size (nm)","pH","temperature/oC"
    ]
    CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]

    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]
    CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df.columns]

    pd.DataFrame({
        "feature": FEATURE_COLS,
        "type": ["numeric" if c in NUMERIC_COLS else "categorical" for c in FEATURE_COLS]
    }).to_csv(out_dir / "feature_info.csv", index=False, encoding="utf-8-sig")

    # ---- logKcat ----
    df_kcat = df.copy()
    kcat_num = pd.to_numeric(df_kcat[COL_KCAT], errors="coerce")
    df_kcat = df_kcat[kcat_num.notna() & (kcat_num > 0)].copy()
    df_kcat.to_csv(out_dir / "used_data_logKcat.csv", index=False, encoding="utf-8-sig")
    run_target("logKcat", df_kcat, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, y_col=COL_KCAT, out_dir=out_dir)

    print(f"\nSaved to: {out_dir}")


# =========================
# 8) Main
# =========================
def main():
    run_one_dataset("SOD", SOD_PATH)
    run_one_dataset("CAT", CAT_PATH)


if __name__ == "__main__":
    main()


########################################################################
Running dataset: SOD
Input file : C:\Users\86158\Desktop\dataissod.xlsx
Output dir : C:\Users\86158\Desktop\sod_kcat_reg_doi_multistage_out
########################################################################
[OK] Output directory writable: C:\Users\86158\Desktop\sod_kcat_reg_doi_multistage_out
[OK] Write test passed for txt/csv/json/joblib

[logKcat] n_samples=10 | n_groups=4 | singleton_groups=1
[logKcat] Stage1 shortlist -> ['KernelRidge_RBF']

[BEST] logKcat | stage=2 | model=KernelRidge_RBF
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 12, 'svd_dim_req': None}
model_params: {'alpha': 0.01, 'gamma': 0.01}
metrics_oof: {'r2_log_oof': 0.5834460919241955, 'mae_log_oof': 1.7713983210061948, 'rmse_log_oof': 2.111071698848368, 'pearson_r_oof': 0.8119883606635298, 'pearson_r2_oof': 0.6593250978530465}

Saved to: C:\Users\86158\Desktop\sod_kcat_reg_doi_multistage_out

#############################

In [2]:
# -*- coding: utf-8 -*-
"""
Plot evaluation figures for SOD/CAT logKcat regression using saved OOF predictions.

Reads:
- <OUT_DIR>/logKcat_oof_predictions_groupkfold.csv

Writes:
- <OUT_DIR>/plots_logKcat/*.png
"""

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ====== EDIT HERE (if needed) ======
SOD_OUT_DIR = Path(r"C:\Users\86158\Desktop\sod_kcat_reg_doi_multistage_out")
CAT_OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_kcat_reg_doi_multistage_out")
TARGET = "logKcat"
# ================================


def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def compute_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "pearson_r": r,
        "pearson_r2": (r * r if not np.isnan(r) else np.nan)
    }


def set_clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


def save_fig(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=260)
    plt.close(fig)


def qq_plot(ax, residuals):
    """
    QQ plot without scipy:
    Uses erfinv if available; otherwise uses a rough linear z grid.
    """
    res = np.asarray(residuals, float)
    res = res[np.isfinite(res)]
    res = np.sort(res)
    n = len(res)
    if n < 6:
        ax.text(0.5, 0.5, f"Too few points for QQ (n={n})",
                ha="center", va="center", transform=ax.transAxes)
        return

    p = (np.arange(1, n + 1) - 0.5) / n

    try:
        from numpy.special import erfinv
        z = np.sqrt(2.0) * erfinv(2.0 * p - 1.0)
    except Exception:
        z = np.linspace(-2.5, 2.5, n)

    ax.scatter(z, res, s=22, alpha=0.75)

    # reference line using quartiles
    q1r, q3r = np.quantile(res, [0.25, 0.75])
    q1z, q3z = np.quantile(z, [0.25, 0.75])
    slope = (q3r - q1r) / (q3z - q1z + 1e-12)
    intercept = q1r - slope * q1z
    xx = np.array([z.min(), z.max()])
    ax.plot(xx, intercept + slope * xx, linestyle="--", linewidth=1)

    ax.set_xlabel("Theoretical quantiles (N(0,1))")
    ax.set_ylabel("Residual quantiles")
    ax.set_title("Residual Q-Q plot")


def plot_one(out_dir: Path, dataset_name: str):
    oof_path = out_dir / f"{TARGET}_oof_predictions_groupkfold.csv"
    if not oof_path.exists():
        raise FileNotFoundError(f"OOF file not found: {oof_path}")

    df = pd.read_csv(oof_path, encoding="utf-8-sig")
    for col in ["y_log_true", "y_log_oof_pred"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["y_log_true", "y_log_oof_pred"]).reset_index(drop=True)

    y_true = df["y_log_true"].values
    y_pred = df["y_log_oof_pred"].values
    resid = y_pred - y_true

    m = compute_metrics(y_true, y_pred)

    plot_dir = out_dir / "plots_logKcat"
    plot_dir.mkdir(parents=True, exist_ok=True)

    # 1) Parity (log10)
    fig = plt.figure(figsize=(6.8, 5.5))
    ax = fig.add_subplot(111)
    ax.scatter(y_true, y_pred, s=30, alpha=0.78)

    mn = float(min(y_true.min(), y_pred.min()))
    mx = float(max(y_true.max(), y_pred.max()))
    pad = 0.06 * (mx - mn + 1e-9)
    lo, hi = mn - pad, mx + pad
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel("True log10(Kcat/s-1)")
    ax.set_ylabel("OOF Pred log10(Kcat/s-1)")
    ax.set_title(f"{dataset_name} logKcat Parity (DOI-GroupKFold OOF)")

    ax.text(
        0.02, 0.98,
        f"R2={m['r2']:.4f}\nRMSE={m['rmse']:.4f}\nMAE={m['mae']:.4f}\nPearson r={m['pearson_r']:.4f}\n(n={len(y_true)})",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "parity_log10.png")

    # 2) Residual vs True
    fig = plt.figure(figsize=(6.8, 5.5))
    ax = fig.add_subplot(111)
    ax.scatter(y_true, resid, s=30, alpha=0.78)
    ax.axhline(0.0, linestyle="--", linewidth=1)

    ax.set_xlabel("True log10(Kcat/s-1)")
    ax.set_ylabel("Residual = Pred - True (log10)")
    ax.set_title(f"{dataset_name} Residual vs True (OOF)")
    ax.text(
        0.02, 0.98,
        f"mean(resid)={np.mean(resid):.4f}\nstd(resid)={np.std(resid, ddof=1):.4f}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "residual_vs_true.png")

    # 3) Residual histogram
    fig = plt.figure(figsize=(6.8, 5.5))
    ax = fig.add_subplot(111)
    ax.hist(resid, bins=10)
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Residual = Pred - True (log10)")
    ax.set_ylabel("Count")
    ax.set_title(f"{dataset_name} Residual Histogram (OOF)")
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "residual_hist.png")

    # 4) QQ residual
    fig = plt.figure(figsize=(6.8, 5.5))
    ax = fig.add_subplot(111)
    qq_plot(ax, resid)
    set_clean_axes(ax)
    save_fig(fig, plot_dir / "qq_residual.png")

    print(f"[Saved plots] {dataset_name}: {plot_dir}")


def main():
    plot_one(SOD_OUT_DIR, "SOD")
    plot_one(CAT_OUT_DIR, "CAT")


if __name__ == "__main__":
    main()

[Saved plots] SOD: C:\Users\86158\Desktop\sod_kcat_reg_doi_multistage_out\plots_logKcat
[Saved plots] CAT: C:\Users\86158\Desktop\cat_kcat_reg_doi_multistage_out\plots_logKcat
